# CS-781 Phase 2 (L-NER) -- E5 Partial / layer-wise FT

**E5: Partial / layer-wise FT** on all four Phase-1 models (order M4, M2, M3, M1). Accelerator: **GPU T4 x2**. Each model: pre-flight (2-GPU smoke test, falls back to 1 GPU), overfit-64 check (first model), LR probe, training with 30-minute checkpoints, dev report, validated `submission.zip`. Then best-of-4 into `phase2_out/E5_partial_ft/BEST/`.

To split the four models across two accounts, change the run cell to e.g. `--models M4,M2` here and `--models M3,M1` on the other account.

**Attach these inputs** (Add Input -> Datasets / Notebook Output):
1. The four Phase-1 outputs (each has `final/model.pt`, `final/config.json`, `final/bpe_model.json`).
   Models are recognised by the `arch` field in `config.json`, so dataset names do not matter.
2. The official Phase-2 files (`Phase_2_LNER`: `fold1/2-text.json`, `fold1/2-gold.json`,
   `labels.txt`, `fold3-text.json`) as a Kaggle dataset. *Alternative:* a read-only Hugging Face
   token as the Kaggle secret `HF`; the files are then downloaded once with plain `urllib`
   (nothing is ever written to the Hub).
3. Optional: the output of `00_data_and_tests` (the `phase2_data` cache) -- otherwise the cache is
   rebuilt here in about a minute on CPU.
4. Optional: the output of an earlier session of THIS notebook: finished models are skipped and an
   unfinished one resumes from its `last.pt`.


In [ ]:
import os, sys, json, hashlib, subprocess, glob
os.makedirs("/kaggle/working/phase2/tests", exist_ok=True)
print(sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(),
          [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as e:
    print("torch not importable:", e)
print("inputs:", sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "none")
EXPECTED_MD5 = {
 "bpe_tokenizer.py": "906d8a37d55bf0081dd5e8d7944c0b4e",
 "phase1_models.py": "1ae9785176b01f67a305e1504b0f7ae6",
 "ner_common.py": "020d134c97ddb4ef8fc97810501a107c",
 "ner_tok.py": "e44b1afb371fc03a00b89af449deacb5",
 "ner_data.py": "7cbb0057ae2347ec9abd80ef3d2e1773",
 "ner_eval.py": "d9a9254e1197ad92f0ae8cdf55385db7",
 "ner_crf.py": "b354358e193b0ccdc14e9cb1668b7dc4",
 "ner_models.py": "6e3e619bf13a1248284f5d75184923e0",
 "ner_peft.py": "d418dc954f786d538ba288e4f34ca454",
 "ner_predict.py": "8b89b2a6eb9e3db19fe5eb5e902d425a",
 "ner_train.py": "31f657fd7401d368f92195484c6257d0",
 "validate_submission.py": "c0b28f7b0bc988d98b683ba0f20ad969",
 "select_best.py": "a1f14966a4e8c0e80a66da146c7f8061",
 "aggregate.py": "050ab23f70a8458a4bc205a1c69cb2ab",
 "run_experiment.py": "3284e4527dbdd9315de3e492cb096261"
}


In [ ]:
%%writefile /kaggle/working/phase2/bpe_tokenizer.py
"""
bpe_tokenizer.py -- from-scratch byte-level BPE (pure Python, stdlib + optional numpy/torch).

No `tokenizers`, `transformers`, `sentencepiece`, `tiktoken` or any other tokenizer library.
Only: re, heapq, json, collections, random, time, gc.

Token-id layout (matches the original notebook's special-token ids):
    0 <|endoftext|>   1 <|pad|>   2 <|unk|>   3..258 = raw bytes 0..255   259.. = merges
Internally training works with "internal ids" (byte b -> b, merge k -> 256+k);
the public API adds NUM_SPECIAL (=3) to every id.
"""
import gc
import heapq
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

SPECIALS = ["<|endoftext|>", "<|pad|>", "<|unk|>"]
NUM_SPECIAL = len(SPECIALS)

# GPT-2 style pre-tokenisation, expressed with the stdlib `re` module
# (\p{L} -> [^\W\d_],  \p{N} -> \d,  punctuation/symbols -> [^\s\w] or '_').
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)"      # English contractions
    r"| ?[^\W\d_]+"               # optional space + letters
    r"| ?\d+"                     # optional space + digits
    r"| ?(?:[^\s\w]|_)+"          # optional space + punctuation / symbols
    r"|\s+(?!\S)"                 # whitespace run not followed by a non-space
    r"|\s+"                       # remaining whitespace
)

MAX_PRETOKEN_CHARS = 512          # pathological runs (e.g. 50k dashes) are cut into pieces


# =============================================================================
# 1. TRAINING
# =============================================================================
def count_pretokens(texts, max_unique=6_000_000, max_word_chars=64, log_every=20000,
                    log=print):
    """Stream documents -> Counter of pre-tokens, with a hard memory ceiling.

    When the Counter grows past `max_unique`, singletons (then rarer and rarer words) are
    pruned, so RAM stays bounded no matter how large the corpus is.
    """
    counter = Counter()
    findall = PRETOKEN_RE.findall
    n_docs = n_chars = 0
    t0 = time.time()
    for text in texts:
        counter.update(findall(text))
        n_docs += 1
        n_chars += len(text)
        if len(counter) > max_unique:
            thr = 2
            while len(counter) > int(max_unique * 0.7):
                counter = Counter({k: v for k, v in counter.items() if v >= thr})
                thr += 1
            gc.collect()
            log(f"   [count] pruned to {len(counter):,} unique pre-tokens (min count {thr - 1})")
        if n_docs % log_every == 0:
            log(f"   [count] {n_docs:,} docs | {n_chars/1e6:,.0f}M chars | "
                f"{len(counter):,} unique | {time.time()-t0:.0f}s")
    # drop absurdly long pre-tokens (junk); they carry no useful merge statistics
    if max_word_chars:
        for k in [k for k in counter if len(k) > max_word_chars]:
            del counter[k]
    return counter


def train_bpe(word_counts, vocab_size, max_words=2_000_000, min_word_count=2,
              min_pair_count=2, n_verify=5000, seed=0, log_every=1000, log=print):
    """Learn BPE merges from a Counter of pre-tokens.

    Classic incremental algorithm: exact pair counts + inverted index (pair -> words that
    contain it) + lazy max-heap, so each merge only touches the words that contain the pair.

    Returns (merges, verify) where
        merges : list[(a, b)] in rank order, in INTERNAL ids (byte b -> b, merge k -> 256+k)
        verify : list[(pretoken_str, final_internal_ids)] used to cross-check the encoder
    """
    num_merges = vocab_size - NUM_SPECIAL - 256
    assert num_merges > 0, "vocab_size too small"

    # ---- 1. pick the training vocabulary of words ---------------------------------
    items = [(w, c) for w, c in word_counts.items() if c >= min_word_count]
    if len(items) > max_words:
        items.sort(key=lambda kv: kv[1], reverse=True)
        items = items[:max_words]
    words, freqs, strs = [], [], []
    for w, c in items:
        ids = list(w.encode("utf-8", "replace"))
        if len(ids) > 1:
            words.append(ids); freqs.append(c); strs.append(w)
    del items
    log(f"   [bpe] training on {len(words):,} unique words "
        f"({sum(freqs)/1e6:,.1f}M word occurrences)")

    # ---- 2. initial pair statistics ------------------------------------------------
    pc = {}        # pair -> total count
    where = {}     # pair -> set(word index)
    for idx, ids in enumerate(words):
        f = freqs[idx]
        for p in zip(ids, ids[1:]):
            if p in pc:
                pc[p] += f
                where[p].add(idx)
            else:
                pc[p] = f
                where[p] = {idx}
    heap = [(-c, p) for p, c in pc.items()]
    heapq.heapify(heap)

    merges = []
    t0 = time.time()
    next_id = 256
    while next_id < 256 + num_merges:
        # ---- pick the most frequent pair (lazy heap: skip stale entries) -----------
        best = None
        while heap:
            negc, p = heapq.heappop(heap)
            if pc.get(p, 0) == -negc:
                best = (-negc, p)
                break
        if best is None:
            log("   [bpe] no more pairs to merge -- stopping early")
            break
        count, (a, b) = best
        if count < min_pair_count:
            log(f"   [bpe] best pair count {count} < {min_pair_count} -- stopping early")
            break

        merges.append((a, b))
        new = next_id
        next_id += 1

        # ---- apply the merge to every word that contains it ------------------------
        idxs = where.pop((a, b), ())
        changed = {}                      # pair -> count before this merge
        for idx in idxs:
            ids = words[idx]
            n = len(ids)
            out = []
            i = 0
            hit = False
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2; hit = True
                else:
                    out.append(ids[i]); i += 1
            if not hit:
                continue
            f = freqs[idx]
            old_pairs = list(zip(ids, ids[1:]))
            new_pairs = list(zip(out, out[1:]))
            for q in old_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] -= f
            for q in new_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] = pc.get(q, 0) + f
            new_set = set(new_pairs)
            for q in set(old_pairs) - new_set:
                s = where.get(q)
                if s is not None:
                    s.discard(idx)
            for q in new_set:
                s = where.get(q)
                if s is None:
                    where[q] = {idx}
                else:
                    s.add(idx)
            words[idx] = out

        # ---- publish the changed pair counts ---------------------------------------
        for q, orig in changed.items():
            c = pc.get(q, 0)
            if c <= 0:
                pc.pop(q, None)
                where.pop(q, None)
            elif c != orig:
                heapq.heappush(heap, (-c, q))
        # keep the heap from growing without bound
        if len(heap) > max(4_000_000, 8 * len(pc)):
            heap = [(-c, q) for q, c in pc.items()]
            heapq.heapify(heap)

        k = len(merges)
        if k % log_every == 0 or k == num_merges:
            log(f"   [bpe] merge {k:,}/{num_merges:,} | last pair count {count:,} | "
                f"{len(pc):,} live pairs | {time.time()-t0:.0f}s")

    # ---- 3. sample of final segmentations (to verify the encoder) ----------------
    rng = random.Random(seed)
    pick = rng.sample(range(len(words)), min(n_verify, len(words)))
    verify = [(strs[i], list(words[i])) for i in pick]
    del words, freqs, strs, pc, where, heap
    gc.collect()
    return merges, verify


# =============================================================================
# 2. TOKENIZER (encode / decode / save / load)
# =============================================================================
class BPETokenizer:
    """Byte-level BPE tokenizer with a small HuggingFace-like surface:
    tok(text, add_special_tokens=False)["input_ids"], tok.encode, tok.decode,
    len(tok), eos_token_id / pad_token_id, padding_side / truncation_side.
    """

    def __init__(self, merges, cache_max=2_000_000):
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = NUM_SPECIAL + 256 + len(self.merges)
        self._merge_map = {m: 256 + r for r, m in enumerate(self.merges)}   # pair -> internal id
        # id -> bytes (internal ids)
        self._bytes = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self._bytes.append(self._bytes[a] + self._bytes[b])
        self._special_bytes = [s.encode("utf-8") for s in SPECIALS]

        self._cache = {}
        self._cache_max = cache_max

        self.eos_token, self.bos_token = SPECIALS[0], SPECIALS[0]
        self.pad_token, self.unk_token = SPECIALS[1], SPECIALS[2]
        self.eos_token_id = self.bos_token_id = 0
        self.pad_token_id = 1
        self.unk_token_id = 2
        self.padding_side = "left"        # decoder-only batched generation
        self.truncation_side = "left"     # an over-long prompt loses its beginning

    # ------------------------------------------------------------------ basics
    def __len__(self):
        return self.vocab_size

    def get_vocab(self):
        v = {s: i for i, s in enumerate(SPECIALS)}
        for i, bs in enumerate(self._bytes):
            v[bs.decode("latin-1") + f"#{i}"] = i + NUM_SPECIAL   # unique readable key
        return v

    # ------------------------------------------------------------------ encoding
    def _bpe_word(self, b):
        """BPE-merge one pre-token given as bytes -> tuple of PUBLIC ids."""
        ids = list(b)
        mm = self._merge_map
        while len(ids) > 1:
            best = None
            bp = None
            for p in zip(ids, ids[1:]):
                m = mm.get(p)
                if m is not None and (best is None or m < best):
                    best, bp = m, p
            if best is None:
                break
            a, c = bp
            out = []
            i, n = 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == c:
                    out.append(best); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return tuple(i + NUM_SPECIAL for i in ids)

    def _encode_pretoken(self, w):
        if len(w) > MAX_PRETOKEN_CHARS:
            res = ()
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                res += self._bpe_word(w[s:s + MAX_PRETOKEN_CHARS].encode("utf-8", "replace"))
            return res
        return self._bpe_word(w.encode("utf-8", "replace"))

    def encode(self, text, add_special_tokens=False):
        cache = self._cache
        out = []
        ext = out.extend
        for w in PRETOKEN_RE.findall(text):
            t = cache.get(w)
            if t is None:
                t = self._encode_pretoken(w)
                if len(cache) >= self._cache_max:
                    cache.clear()                 # bounded RAM
                cache[w] = t
            ext(t)
        return out

    def __call__(self, text, add_special_tokens=False, padding=False, truncation=False,
                 max_length=None, return_tensors=None, **_ignored):
        single = isinstance(text, str)
        batch = [text] if single else list(text)
        enc = [self.encode(t) for t in batch]
        if truncation and max_length is not None:
            if self.truncation_side == "left":
                enc = [e[-max_length:] if len(e) > max_length else e for e in enc]
            else:
                enc = [e[:max_length] for e in enc]
        masks = [[1] * len(e) for e in enc]
        if padding:
            target = max_length if (padding == "max_length" and max_length) else max(len(e) for e in enc)
            for k, e in enumerate(enc):
                gap = target - len(e)
                if gap > 0:
                    if self.padding_side == "left":
                        enc[k] = [self.pad_token_id] * gap + e
                        masks[k] = [0] * gap + masks[k]
                    else:
                        enc[k] = e + [self.pad_token_id] * gap
                        masks[k] = masks[k] + [0] * gap
        if return_tensors in ("pt", "np"):
            if return_tensors == "pt":
                import torch
                mk = lambda x: torch.tensor(x, dtype=torch.long)
            else:
                import numpy as np
                mk = lambda x: np.asarray(x, dtype=np.int64)
            return {"input_ids": mk(enc), "attention_mask": mk(masks)}
        if single:
            return {"input_ids": enc[0], "attention_mask": masks[0]}
        return {"input_ids": enc, "attention_mask": masks}

    # ------------------------------------------------------------------ decoding
    def decode(self, ids, skip_special_tokens=False):
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        nb = len(self._bytes)
        for i in ids:
            i = int(i)
            if i < NUM_SPECIAL:
                if not skip_special_tokens and i >= 0:
                    parts.append(self._special_bytes[i])
            elif i - NUM_SPECIAL < nb:
                parts.append(self._bytes[i - NUM_SPECIAL])
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, seqs, skip_special_tokens=False):
        return [self.decode(s, skip_special_tokens) for s in seqs]

    # ------------------------------------------------------------------ persistence
    def save(self, path):
        obj = {"type": "byte_level_bpe", "version": 1, "specials": SPECIALS,
               "num_special": NUM_SPECIAL, "vocab_size": self.vocab_size,
               "pretoken_regex": PRETOKEN_RE.pattern, "merges": self.merges}
        with open(path, "w") as f:
            json.dump(obj, f)

    def save_pretrained(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.save(out_dir / "bpe_model.json")
        with open(out_dir / "tokenizer_config.json", "w") as f:
            json.dump({"eos_token": self.eos_token, "pad_token": self.pad_token,
                       "unk_token": self.unk_token, "eos_token_id": 0, "pad_token_id": 1,
                       "unk_token_id": 2, "vocab_size": self.vocab_size,
                       "padding_side": self.padding_side,
                       "truncation_side": self.truncation_side}, f, indent=2)

    @classmethod
    def load(cls, path, **kw):
        with open(path) as f:
            obj = json.load(f)
        assert obj.get("type") == "byte_level_bpe", "not a byte_level_bpe file"
        assert obj["pretoken_regex"] == PRETOKEN_RE.pattern, "pre-tokenizer regex mismatch"
        return cls(obj["merges"], **kw)

    @classmethod
    def from_pretrained(cls, directory, **kw):
        return cls.load(Path(directory) / "bpe_model.json", **kw)


In [ ]:
%%writefile /kaggle/working/phase2/phase1_models.py
"""
phase1_models.py -- the Phase-1 model code, copied VERBATIM from the Phase-1 training notebooks.

  * cell 13 of every notebook (identical in all four): shared core
    (CausalLMOutput, positions_from_mask, causal_allowed, build_attn_mask, init_weights,
     decoding helpers, CausalLMBase). The only change: its trailing print() was removed so
     that importing this module is silent.
  * cell 15, class definitions only (everything before `MODEL_CLASS, CONFIG_CLASS = ...`):
      - 01_train_gpt2style       -> GPT2StyleConfig, GPT2Attention, GELUMLP, GPT2Block, GPT2StyleLM
      - 02_train_llamastyle      -> LlamaStyleConfig, RMSNorm, RotaryEmbedding, GQAttention, SwiGLU,
                                    LlamaBlock, LlamaStyleLM
        (03_train_mistral_sliding defines the SAME classes; it differs only by a module-level
         SLIDING_WINDOW constant and one comment. The window comes from config.json, so one
         copy serves both M2 and M3.)
      - 04_train_bloom_alibi     -> BloomStyleConfig, alibi_slopes, alibi_bias, BloomAttention,
                                    BloomMLP, BloomBlock, BloomStyleLM

DO NOT EDIT the copied code: state_dict keys of the Phase-1 `model.pt` files depend on it.
Everything NER-specific lives in ner_models.py.
"""

# =============================================================================================
# notebook cell 13 (shared core)
# =============================================================================================
import math
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view


@dataclass
class CausalLMOutput:
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[list] = None      # per layer: (k, v), each (B, n_kv_heads, S, head_dim)


def positions_from_mask(attention_mask, past_len, T, B, device):
    """Position ids for the T new tokens. attention_mask (if given) covers ALL keys: (B, past_len+T)."""
    if attention_mask is None:
        return torch.arange(past_len, past_len + T, device=device).unsqueeze(0).expand(B, T)
    pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
    return pos[:, -T:]


def causal_allowed(T, past_len, attention_mask, window, device):
    """Bool (B|1, 1, T, S): True where query t may attend key s. Always materialised."""
    S = past_len + T
    q = torch.arange(past_len, S, device=device)[:, None]      # absolute index of each query
    k = torch.arange(S, device=device)[None, :]
    allowed = k <= q                                             # causal
    if window is not None:
        allowed = allowed & (q - k < window)                     # sliding window: last `window` keys incl. self
    allowed = allowed[None, None]
    if attention_mask is not None:
        allowed = allowed & attention_mask[:, None, None, :].bool()   # drop padding keys
        allowed = allowed | (q == k)[None, None]                 # a pad query would see nothing -> NaN; let it see itself
    return allowed


def build_attn_mask(T, past_len, attention_mask, window, device):
    """Returns (mask, is_causal) for F.scaled_dot_product_attention."""
    S = past_len + T
    window_active = window is not None and window < S
    no_pad = attention_mask is None or bool(attention_mask.all())
    if no_pad and not window_active:
        if past_len == 0:
            return None, True        # fast fused causal kernel (training path)
        if T == 1:
            return None, False       # single cached query sees every key
    return causal_allowed(T, past_len, None if no_pad else attention_mask,
                          window if window_active else None, device), False


def init_weights(model, n_layer, std=0.02):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():       # residual-branch output projections
        if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layer))


# ------------------------------------------------------------------ decoding helpers
def _sample(logits, temperature, top_p, top_k):
    logits = logits / max(float(temperature), 1e-5)
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(int(top_k), logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        remove = probs.cumsum(-1) - probs > top_p           # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _banned_ngram_tokens(row, n):
    """Tokens that would complete an n-gram already present in `row` (list of ints)."""
    if n < 2 or len(row) < n:
        return []
    arr = np.asarray(row)
    win = sliding_window_view(arr[:-1], n - 1)           # window i = arr[i:i+n-1], followed by arr[i+n-1]
    hit = np.nonzero((win == arr[-(n - 1):]).all(1))[0]
    return arr[hit + n - 1].tolist()


class CausalLMBase(nn.Module):
    """Loss + generation shared by all four architectures."""

    @staticmethod
    def lm_loss(logits, labels):
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                               labels[:, 1:].reshape(-1), ignore_index=-100)

    def _process_logits(self, logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress):
        if repetition_penalty and repetition_penalty != 1.0:
            sc = torch.gather(logits, 1, seq)
            sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
            logits = logits.scatter(1, seq, sc)
        if no_repeat_ngram_size and no_repeat_ngram_size > 1:
            for b, row in enumerate(rows):
                banned = _banned_ngram_tokens(row, no_repeat_ngram_size)
                if banned:
                    logits[b, banned] = float("-inf")
        if suppress:
            logits[:, list(suppress)] = float("-inf")
        return logits

    @torch.inference_mode()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=256, do_sample=False,
                 temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                 no_repeat_ngram_size=0, num_beams=1, suppress_token_ids=(),
                 stop_fn=None, stop_check_every=16, **_ignored):
        """Returns ONLY the new tokens, shape (B, n_new). stop_fn(new_tokens) -> True ends early."""
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        attention_mask = attention_mask.long()
        rows = [ids[m.bool()].tolist() for ids, m in zip(input_ids, attention_mask)]   # real tokens only
        if num_beams > 1:
            return self._beam_search(input_ids, attention_mask, rows, max_new_tokens, num_beams,
                                     repetition_penalty, no_repeat_ngram_size, suppress_token_ids,
                                     stop_fn, stop_check_every)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
        seq, new = input_ids, []
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty,
                                          no_repeat_ngram_size, suppress_token_ids)
            nxt = _sample(logits, temperature, top_p, top_k) if do_sample else logits.argmax(-1)
            new.append(nxt)
            seq = torch.cat([seq, nxt[:, None]], 1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(nxt)[:, None]], 1)
            for row, t in zip(rows, nxt.tolist()):
                row.append(t)
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(torch.stack(new, 1)):
                break
            out = self(nxt[:, None], attention_mask=attention_mask, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        return torch.stack(new, 1)

    def _beam_search(self, input_ids, attention_mask, rows, max_new_tokens, K, repetition_penalty,
                     no_repeat_ngram_size, suppress, stop_fn, stop_check_every):
        B = input_ids.size(0)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in out.past_key_values]
        logits = out.logits[:, -1].float().repeat_interleave(K, 0)
        seq = input_ids.repeat_interleave(K, 0)
        am = attention_mask.repeat_interleave(K, 0)
        rows = [list(r) for r in rows for _ in range(K)]
        scores = torch.zeros(B, K, device=input_ids.device)
        scores[:, 1:] = -1e9                       # all beams start identical: expand from beam 0 only
        gen = seq[:, :0]
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress)
            logp = logits.log_softmax(-1)
            V = logp.size(-1)
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            gen = torch.cat([gen[src], tok[:, None]], 1)
            seq = torch.cat([seq[src], tok[:, None]], 1)
            am = torch.cat([am[src], torch.ones_like(tok)[:, None]], 1)
            rows = [rows[i] + [t] for i, t in zip(src.tolist(), tok.tolist())]
            past = [(k[src], v[src]) for k, v in past]
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(gen):
                break
            out = self(tok[:, None], attention_mask=am, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        best = scores.argmax(1)
        return gen.view(B, K, -1)[torch.arange(B, device=best.device), best]


# =============================================================================================
# 01_train_gpt2style, cell 15 (classes)
# =============================================================================================
@dataclass
class GPT2StyleConfig:
    vocab_size: int
    n_positions: int = 2048
    n_embd: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


class GPT2Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.n_head, self.head_dim, self.d = c.n_head, c.n_embd // c.n_head, c.n_embd
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.o_proj = nn.Linear(c.n_embd, c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(self.d, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # (B, H, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        y = y.transpose(1, 2).reshape(B, T, self.d)                        # merge heads
        return self.o_proj(y), (k, v)


class GELUMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.attn = GPT2Attention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.mlp = GELUMLP(c.n_embd, c.mlp_ratio * c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        a, present = self.attn(self.ln_1(x), mask, is_causal, past)       # pre-LN residual
        x = x + a
        x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2StyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.wpe = nn.Embedding(c.n_positions, c.n_embd)
        self.blocks = nn.ModuleList([GPT2Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.wte.weight                              # tied input/output embeddings

    def get_input_embeddings(self):
        return self.wte

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        pos = pos.clamp(max=self.config.n_positions - 1)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask, None, input_ids.device)
        x = self.wte(input_ids) + self.wpe(pos)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 02_train_llamastyle, cell 15 (classes) -- also used for 03_train_mistral_sliding
# =============================================================================================
@dataclass
class LlamaStyleConfig:
    vocab_size: int
    max_position_embeddings: int = 2048
    hidden_size: int = 768
    intermediate_size: int = 3072
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    num_key_value_heads: int = 4
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    sliding_window: Optional[int] = None      # None = full causal attention
    initializer_range: float = 0.02


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, pos):                                   # pos: (B, T) int
        freqs = pos[..., None].float() * self.inv_freq        # (B, T, hd/2)
        emb = torch.cat([freqs, freqs], dim=-1)               # (B, T, hd): angle for dim i and i+hd/2
        return emb.cos()[:, None], emb.sin()[:, None]         # (B, 1, T, hd) -> broadcast over heads


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return (x * cos + rotate_half(x) * sin).to(x.dtype)


class GQAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.num_attention_heads == 0
        assert c.num_attention_heads % c.num_key_value_heads == 0, "KV heads must divide query heads"
        self.h, self.kv = c.num_attention_heads, c.num_key_value_heads
        self.hd = c.hidden_size // c.num_attention_heads
        self.n_rep = self.h // self.kv
        self.q_proj = nn.Linear(c.hidden_size, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, c.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)    # (B, 12, T, hd)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)   # (B,  4, T, hd)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)           # rotate Q and K only
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        present = (k, v)                                                  # cache stays at 4 heads
        if self.n_rep > 1:                                                # kv head j serves q heads j*3..j*3+2
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.h * self.hd)), present


class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d, hidden, bias=False)
        self.up_proj = nn.Linear(d, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class LlamaBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.attn = GQAttention(c)
        self.post_attn_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.mlp = SwiGLU(c.hidden_size, c.intermediate_size)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        a, present = self.attn(self.input_norm(x), cos, sin, mask, is_causal, past)
        x = x + a
        x = x + self.mlp(self.post_attn_norm(x))
        return x, present


class LlamaStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embed_tokens = nn.Embedding(c.vocab_size, c.hidden_size)
        self.rotary = RotaryEmbedding(c.hidden_size // c.num_attention_heads, c.rope_theta)
        self.layers = nn.ModuleList([LlamaBlock(c) for _ in range(c.num_hidden_layers)])
        self.norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        init_weights(self, c.num_hidden_layers, c.initializer_range)
        self.lm_head.weight = self.embed_tokens.weight

    def get_input_embeddings(self):
        return self.embed_tokens

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        cos, sin = self.rotary(pos)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask,
                                          self.config.sliding_window, input_ids.device)
        x = self.embed_tokens(input_ids)
        presents = []
        for i, layer in enumerate(self.layers):
            x, pr = layer(x, cos, sin, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.norm(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 04_train_bloom_alibi, cell 15 (classes)
# =============================================================================================
@dataclass
class BloomStyleConfig:
    vocab_size: int
    hidden_size: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


def alibi_slopes(n_head):
    def pow2_slopes(n):
        start = 2.0 ** (-(2.0 ** -(math.log2(n) - 3)))
        return [start * start ** i for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    closest = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(closest) + alibi_slopes(2 * closest)[0::2][: n_head - closest]


def alibi_bias(pos_q, pos_k, slopes, allowed):
    """pos_q (B,T), pos_k (B,S), slopes (H,), allowed bool (B|1,1,T,S) -> float bias (B,H,T,S)."""
    dist = (pos_q[:, None, :, None] - pos_k[:, None, None, :]).clamp(min=0).float()
    bias = -slopes.view(1, -1, 1, 1) * dist
    return bias.masked_fill(~allowed, float("-inf"))


class BloomAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.n_head == 0
        self.h, self.hd, self.d = c.n_head, c.hidden_size // c.n_head, c.hidden_size
        self.query_key_value = nn.Linear(c.hidden_size, 3 * c.hidden_size)
        self.o_proj = nn.Linear(c.hidden_size, c.hidden_size)

    def forward(self, x, bias, past=None):
        B, T, _ = x.shape
        q, k, v = self.query_key_value(x).split(self.d, dim=-1)
        q = q.view(B, T, self.h, self.hd).transpose(1, 2)
        k = k.view(B, T, self.h, self.hd).transpose(1, 2)
        v = v.view(B, T, self.h, self.hd).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))   # additive ALiBi bias
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.d)), (k, v)


class BloomMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class BloomBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.self_attention = BloomAttention(c)
        self.post_attention_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.mlp = BloomMLP(c.hidden_size, c.mlp_ratio * c.hidden_size)

    def forward(self, x, bias, past=None):
        a, present = self.self_attention(self.input_layernorm(x), bias, past)
        x = x + a
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x, present


class BloomStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.word_embeddings = nn.Embedding(c.vocab_size, c.hidden_size)
        self.blocks = nn.ModuleList([BloomBlock(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        self.register_buffer("slopes", torch.tensor(alibi_slopes(c.n_head), dtype=torch.float32),
                             persistent=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.word_embeddings.weight

    def get_input_embeddings(self):
        return self.word_embeddings

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        dev = input_ids.device
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        S = past_len + T
        if attention_mask is None:                          # all rows identical -> batch dim 1
            pos_k = torch.arange(S, device=dev)[None]
            allowed = causal_allowed(T, past_len, None, None, dev)
        else:
            pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
            allowed = causal_allowed(T, past_len, attention_mask, None, dev)
        bias = alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)    # computed once, shared by all layers
        x = self.word_embeddings(input_ids)                  # no position embedding, no embedding LayerNorm (see note)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, bias, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# ---------------------------------------------------------------------------------------------
# Registry (added for Phase 2; not part of the copied code)
# ---------------------------------------------------------------------------------------------
MODEL_CLASSES = {
    "GPT2StyleLM": (GPT2StyleLM, GPT2StyleConfig),
    "LlamaStyleLM": (LlamaStyleLM, LlamaStyleConfig),
    "BloomStyleLM": (BloomStyleLM, BloomStyleConfig),
}


In [ ]:
%%writefile /kaggle/working/phase2/ner_common.py
"""
ner_common.py -- small torch-free helpers shared by every Phase-2 module.

  * registries: the four Phase-1 models (M1..M4) and the experiments (E1..E6, extras)
  * file discovery under /kaggle/input (or any root), atomic JSON writes
  * disk guard and output-size accounting (no Hugging Face Hub anywhere in this project)
"""
import glob
import hashlib
import json
import os
import shutil
import time

# ---------------------------------------------------------------------------------------------
# Registries
# ---------------------------------------------------------------------------------------------
# `arch` must equal the "arch" field the Phase-1 notebooks wrote into final/config.json.
MODELS = {
    "M1": {"arch": "gpt2style", "folder": "M1_gpt2style", "kind": "gpt2"},
    "M2": {"arch": "llamastyle", "folder": "M2_llamastyle", "kind": "llama"},
    "M3": {"arch": "mistral-sliding", "folder": "M3_mistral_sliding", "kind": "llama"},
    "M4": {"arch": "bloom-alibi", "folder": "M4_bloom_alibi", "kind": "bloom"},
}
DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]          # strongest Phase-1 LM first

# Layer subsets for the layer-pruned KD students (every second layer; section 8 of the brief).
STUDENT_LAYERS = {
    "M1": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
    "M2": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M3": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M4": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
}

# The six required experiments + extras. `technique` selects the code path in ner_peft.py.
EXPERIMENTS = {
    "E1": {"folder": "E1_full_ft", "technique": "full_ft", "title": "Full fine-tuning"},
    "E2": {"folder": "E2_lorma", "technique": "lorma", "title": "LoRMA+ (compulsory)"},
    "E3": {"folder": "E3_lora", "technique": "lora", "title": "LoRA"},
    "E4": {"folder": "E4_adapters", "technique": "adapters", "title": "Houlsby adapters"},
    "E5": {"folder": "E5_partial_ft", "technique": "partial_ft", "title": "Partial / layer-wise FT"},
    "E6": {"folder": "E6_kd", "technique": "kd", "title": "Knowledge distillation"},
    # extras (not counted among the six)
    "X1": {"folder": "X1_bitfit", "technique": "bitfit", "title": "BitFit (M1/M4 only)",
           "models": ["M4", "M1"]},
    "X2": {"folder": "X2_student_no_kd", "technique": "student_hard",
           "title": "Pruned student, hard labels only (KD control)"},
}
REQUIRED_EXPS = ["E1", "E2", "E3", "E4", "E5", "E6"]

# Default learning rates (section 9.3). The LR probe multiplies `lr` by {1/3, 1, 3}.
DEFAULT_LR = {
    "full_ft": 5e-5, "lorma": 1e-3, "lora": 1e-3, "adapters": 1e-3, "bitfit": 2e-3,
    "partial_ft": 1e-4, "kd": 1e-4, "student_hard": 1e-4,
}
DEFAULT_EPOCHS = {
    "full_ft": 6, "kd": 6, "student_hard": 6, "partial_ft": 8,
    "lorma": 10, "lora": 10, "adapters": 10, "bitfit": 10,
}


# ---------------------------------------------------------------------------------------------
# JSON / files
# ---------------------------------------------------------------------------------------------
def write_json(path, obj, indent=2):
    """Atomic JSON write: a crash never leaves a half-written file behind."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    try:
        import numpy as np
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
    except ImportError:
        pass
    raise TypeError(f"not JSON serialisable: {type(o)}")


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def find_files(name, roots, max_hits=50):
    """All files called `name` below any of `roots` (recursive), sorted for determinism."""
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            hits.extend(glob.glob(os.path.join(r, "**", name), recursive=True))
    return sorted(set(hits))[:max_hits]


def default_input_roots():
    """Where attached Kaggle datasets live; PHASE2_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE2_INPUT_ROOTS")
    if env:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def iter_finished_runs(out_root, input_roots=()):
    """Yield (exp_folder, model_folder, run_dir) for every finished run (DONE.json):
    directly under `out_root` (<exp>/<model>/DONE.json, whatever out_root is called) and under
    any `phase2_out` folder inside the attached input datasets. First occurrence wins."""
    seen = set()
    out_root = os.path.abspath(out_root) if out_root else None
    if out_root and os.path.isdir(out_root):
        for d in sorted(glob.glob(os.path.join(out_root, "*", "*", "DONE.json"))):
            exp_f, mod_f = d.split(os.sep)[-3], d.split(os.sep)[-2]
            if mod_f != "BEST" and (exp_f, mod_f) not in seen:
                seen.add((exp_f, mod_f))
                yield exp_f, mod_f, os.path.dirname(d)
    for d in find_files("DONE.json", list(input_roots), max_hits=5000):
        parts = d.split(os.sep)
        if "phase2_out" not in parts:
            continue
        i = len(parts) - 1 - parts[::-1].index("phase2_out")
        if len(parts) - i != 4 or parts[-2] == "BEST":
            continue
        if (parts[i + 1], parts[-2]) not in seen:
            seen.add((parts[i + 1], parts[-2]))
            yield parts[i + 1], parts[-2], os.path.dirname(d)


def find_phase1_models(roots=None):
    """Map M1..M4 -> directory holding that model's Phase-1 `model.pt` + `config.json`.

    Identified by the "arch" field the Phase-1 notebooks wrote into config.json, so the
    dataset/folder names do not matter.
    """
    roots = roots or default_input_roots()
    by_arch = {}
    for cfg_path in find_files("config.json", roots, max_hits=10000):
        d = os.path.dirname(cfg_path)
        if not os.path.exists(os.path.join(d, "model.pt")):
            continue
        try:
            cfg = read_json(cfg_path)
        except Exception:
            continue
        if isinstance(cfg, dict) and "arch" in cfg and "model_class" in cfg:
            by_arch.setdefault(cfg["arch"], []).append(d)
    out = {}
    for mid, m in MODELS.items():
        cands = by_arch.get(m["arch"], [])
        if cands:
            # prefer a directory literally called "final" (what the Phase-1 notebooks saved)
            cands = sorted(cands, key=lambda p: (os.path.basename(p) != "final", p))
            out[mid] = cands[0]
    return out


# ---------------------------------------------------------------------------------------------
# Disk guard (section 9.5)
# ---------------------------------------------------------------------------------------------
def free_gb(path="/kaggle/working"):
    p = path
    while p and not os.path.exists(p):
        p = os.path.dirname(p)
    return shutil.disk_usage(p or "/").free / 1e9


def dir_size_gb(path):
    tot = 0
    for dp, _dn, fns in os.walk(path):
        for fn in fns:
            fp = os.path.join(dp, fn)
            if not os.path.islink(fp):
                try:
                    tot += os.path.getsize(fp)
                except OSError:
                    pass
    return tot / 1e9


class DiskFull(RuntimeError):
    pass


def disk_guard(out_root, min_free_gb=4.0, log=print, keep=None):
    """Make sure at least `min_free_gb` is free under `out_root`.

    Deletion order: leftover last.pt / last.tmp of models that are finished (DONE.json) or not
    the one currently running (`keep` = its checkpoint dir), then tmp files. Never touches
    best.pt, submissions or reports. Raises DiskFull if still short (callers stop cleanly).
    """
    if free_gb(out_root) >= min_free_gb:
        return
    removed = []
    for pat in ("**/checkpoints/last.pt", "**/checkpoints/last.tmp", "**/*.tmp*"):
        for f in glob.glob(os.path.join(out_root, pat), recursive=True):
            if keep and os.path.abspath(f).startswith(os.path.abspath(keep)):
                continue
            try:
                sz = os.path.getsize(f) / 1e9
                os.remove(f)
                removed.append((f, sz))
            except OSError:
                pass
        if free_gb(out_root) >= min_free_gb:
            break
    for f, sz in removed:
        log(f"   [disk] removed {f} ({sz:.2f} GB)")
    if free_gb(out_root) < min_free_gb:
        raise DiskFull(f"only {free_gb(out_root):.1f} GB free under {out_root} "
                       f"(need {min_free_gb} GB) even after cleanup")


# ---------------------------------------------------------------------------------------------
# misc
# ---------------------------------------------------------------------------------------------
def now():
    return time.time()


def fmt_h(seconds):
    return f"{seconds / 3600:.2f} h"


def log_line(path, msg, also_print=True):
    """Append a timestamped line to a text log (and print it)."""
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + msg
    if also_print:
        print(line, flush=True)
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


In [ ]:
%%writefile /kaggle/working/phase2/ner_tok.py
"""
ner_tok.py -- character offsets for the Phase-1 byte-level BPE tokenizer.

bpe_tokenizer.py (copied verbatim, never modified) only returns token ids. NER needs, for every
token, which characters of the ORIGINAL string it covers. This module re-runs exactly the
tokenizer's own steps (regex pre-tokenisation -> 512-char pieces -> BPE merges per piece) and
tracks byte positions, so the ids are guaranteed identical to `tok.encode(text)` (asserted).

Per token we return
  raw span  [raw_start, raw_end)  : the characters the token *owns* (see rule below)
  eff span  [eff_start, eff_end)  : the raw span with leading/trailing whitespace trimmed
                                    (pre-tokens carry a leading space: " Court" -> "Court")
  -1 / -1                         : the token owns no character / is whitespace only

Ownership rule (byte-level BPE can split a multi-byte UTF-8 character across two tokens):
a character is owned by the token that contains its FIRST byte. A token holding only
continuation bytes owns no character (raw span -1, -1); ner_data.py gives such a token the
label of the token before it.
"""
import bisect

import numpy as np

from bpe_tokenizer import MAX_PRETOKEN_CHARS, NUM_SPECIAL, PRETOKEN_RE


class TokenizedDoc:
    """Token ids + character spans for one document (numpy int32 arrays, -1 = none)."""
    __slots__ = ("ids", "raw_start", "raw_end", "eff_start", "eff_end")

    def __init__(self, ids, raw_start, raw_end, eff_start, eff_end):
        self.ids = np.asarray(ids, dtype=np.int32)
        self.raw_start = np.asarray(raw_start, dtype=np.int32)
        self.raw_end = np.asarray(raw_end, dtype=np.int32)
        self.eff_start = np.asarray(eff_start, dtype=np.int32)
        self.eff_end = np.asarray(eff_end, dtype=np.int32)

    def __len__(self):
        return int(self.ids.shape[0])

    def to_dict(self):
        return {k: getattr(self, k) for k in self.__slots__}

    @classmethod
    def from_dict(cls, d):
        return cls(*(d[k] for k in cls.__slots__))


def _piece_tokens(tok, piece):
    """BPE one piece (<= MAX_PRETOKEN_CHARS chars).

    Returns a tuple of (id, raw_s, raw_e, eff_s, eff_e) in PIECE-LOCAL character coordinates.
    """
    b = piece.encode("utf-8", "replace")
    ids = tok._bpe_word(b)                                   # public ids, same call as encode()
    # byte offset at which each character starts ("replace" turns a lone surrogate into b"?",
    # exactly like the whole-string encode above, so lengths agree)
    char_byte_start, pos = [], 0
    for ch in piece:
        char_byte_start.append(pos)
        pos += len(ch.encode("utf-8", "replace"))
    assert pos == len(b), "per-character UTF-8 lengths disagree with the whole-piece encoding"

    out, bpos = [], 0
    for tid in ids:
        nb = len(tok._bytes[tid - NUM_SPECIAL])
        b0, b1 = bpos, bpos + nb
        bpos = b1
        c0 = bisect.bisect_left(char_byte_start, b0)        # first char starting at/after b0
        c1 = bisect.bisect_left(char_byte_start, b1)        # first char starting at/after b1
        if c0 >= c1:                                         # only continuation bytes
            out.append((tid, -1, -1, -1, -1))
            continue
        e0, e1 = c0, c1
        while e0 < e1 and piece[e0].isspace():
            e0 += 1
        while e1 > e0 and piece[e1 - 1].isspace():
            e1 -= 1
        if e0 >= e1:                                         # whitespace-only token
            out.append((tid, c0, c1, -1, -1))
        else:
            out.append((tid, c0, c1, e0, e1))
    assert bpos == len(b), "token bytes do not add up to the piece's bytes"
    return tuple(out)


class OffsetEncoder:
    """Wraps a BPETokenizer; caches per-pre-token results (pre-tokens repeat a lot)."""

    def __init__(self, tok, cache_max=500_000):
        self.tok = tok
        self._cache = {}
        self._cache_max = cache_max

    def _pretoken(self, w):
        hit = self._cache.get(w)
        if hit is not None:
            return hit
        # identical piece rule to BPETokenizer._encode_pretoken
        if len(w) > MAX_PRETOKEN_CHARS:
            res, base = [], 0
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                for (tid, a, b, c, d) in _piece_tokens(self.tok, w[s:s + MAX_PRETOKEN_CHARS]):
                    shift = lambda v: v + s if v >= 0 else -1
                    res.append((tid, shift(a), shift(b), shift(c), shift(d)))
            res = tuple(res)
        else:
            res = _piece_tokens(self.tok, w)
        if len(self._cache) >= self._cache_max:
            self._cache.clear()
        self._cache[w] = res
        return res

    def encode(self, text, check=True):
        ids, rs, re_, es, ee = [], [], [], [], []
        for m in PRETOKEN_RE.finditer(text):
            base = m.start()
            for (tid, a, b, c, d) in self._pretoken(m.group()):
                ids.append(tid)
                rs.append(a + base if a >= 0 else -1)
                re_.append(b + base if b >= 0 else -1)
                es.append(c + base if c >= 0 else -1)
                ee.append(d + base if d >= 0 else -1)
        if check:
            ref = self.tok.encode(text)
            if ref != ids:
                raise AssertionError("offset encoder produced different ids than tok.encode()")
        return TokenizedDoc(ids, rs, re_, es, ee)


def encode_with_offsets(tok, text, check=True):
    """Convenience one-shot API (builds a throw-away cache)."""
    return OffsetEncoder(tok).encode(text, check=check)


def verify_offsets(text, td):
    """Raw spans must partition the text in order: every character owned exactly once.

    Returns the number of problems found (0 = OK). Also checks that eff spans lie inside raw
    spans and contain no leading/trailing whitespace.
    """
    problems = 0
    nxt = 0
    for i in range(len(td)):
        a, b = int(td.raw_start[i]), int(td.raw_end[i])
        if a < 0:
            continue
        if a != nxt or b <= a:
            problems += 1
        nxt = b
        c, d = int(td.eff_start[i]), int(td.eff_end[i])
        if c >= 0:
            if not (a <= c < d <= b) or text[c].isspace() or text[d - 1].isspace():
                problems += 1
        elif not text[a:b].isspace():
            problems += 1
    if nxt != len(text):
        problems += 1
    return problems


In [ ]:
%%writefile /kaggle/working/phase2/ner_data.py
"""
ner_data.py -- Phase-2 data: loading, gates G0/G1/G2, IOB2 tagging, chunking, dev split, cache.

Torch-free (numpy + stdlib). Run on CPU:

    python ner_data.py build --out /kaggle/working/phase2_data

Gates (section 11 of the brief):
  G0  every gold entity satisfies text[start:end] == entity_text  (end exclusive)
  G1  gold spans -> token tags -> decoded spans round trip ("ceiling") strict F1 >= 0.97 (real data: 0.9825, the rest is gold ending mid-BPE-token, unreachable by any token tagger)
  G2  token tags -> chunks -> overlap merge -> decode gives exactly the G1 spans

Text is NEVER normalised: offsets are Python str indices into the raw document string.
"""
import argparse
import json
import os
import pickle
import random
import sys
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict

import numpy as np

from ner_common import default_input_roots, file_md5, find_files, read_json, write_json

EXPECTED_LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT",
                   "STAT", "PREC", "DATE", "CASENO"]
PHASE2_FILES = {
    "fold1_text": "fold1-text.json", "fold1_gold": "fold1-gold.json",
    "fold2_text": "fold2-text.json", "fold2_gold": "fold2-gold.json",
    "labels": "labels.txt", "fold3_text": "fold3-text.json",
}
HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_PATHS = {   # tried in order when the files are not attached as a Kaggle dataset
    "fold1_text": ["Phase_2_LNER/Train/fold1-text.json"],
    "fold1_gold": ["Phase_2_LNER/Train/fold1-gold.json"],
    "fold2_text": ["Phase_2_LNER/Train/fold2-text.json"],
    "fold2_gold": ["Phase_2_LNER/Train/fold2-gold.json"],
    "labels": ["Phase_2_LNER/Train/labels.txt", "Phase_2_LNER/labels.txt"],
    "fold3_text": ["Phase_2_LNER/Test/fold3-text.json", "Phase_2_LNER/Train/fold3-text.json"],
}
IGNORE = -100


class GateFailure(RuntimeError):
    """A 'stop and ask' condition from section 12 of the brief."""


# =============================================================================================
# 1. locating / downloading the raw files
# =============================================================================================
def _hf_read_token():
    """Optional READ token for downloading the official data (never used for writing)."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient          # only exists on Kaggle
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None


def _download(rel_path, dest, token):
    url = f"https://huggingface.co/datasets/{HF_REPO}/resolve/main/{rel_path}"
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=120) as r, open(dest + ".part", "wb") as f:
        while True:
            b = r.read(1 << 20)
            if not b:
                break
            f.write(b)
    os.replace(dest + ".part", dest)


def locate_raw_files(raw_dir, roots=None, allow_download=True, log=print):
    """Find the six Phase-2 files under the input roots, else download them (read-only)."""
    roots = roots or default_input_roots()
    found = {}
    for key, name in PHASE2_FILES.items():
        hits = [h for h in find_files(name, roots + [raw_dir]) if "phase2_out" not in h]
        if hits:
            # prefer paths that look like the official layout
            hits.sort(key=lambda p: ("Phase_2" not in p, len(p)))
            found[key] = hits[0]
    missing = [k for k in PHASE2_FILES if k not in found]
    if missing and allow_download:
        os.makedirs(raw_dir, exist_ok=True)
        token = _hf_read_token()
        for key in missing:
            dest = os.path.join(raw_dir, PHASE2_FILES[key])
            err = None
            for rel in HF_PATHS[key]:
                try:
                    log(f"   downloading {rel} ...")
                    _download(rel, dest, token)
                    found[key] = dest
                    break
                except (urllib.error.URLError, OSError) as e:
                    err = e
            if key not in found:
                log(f"   could not download {PHASE2_FILES[key]}: {err}")
    missing = [PHASE2_FILES[k] for k in PHASE2_FILES if k not in found]
    if missing:
        raise FileNotFoundError(
            "Phase-2 files not found: " + ", ".join(missing) + ". Attach the official "
            "Phase_2_LNER folder as a Kaggle dataset (or add a read-only HF token as the Kaggle "
            "secret 'HF').")
    return found


# =============================================================================================
# 2. tolerant loaders
# =============================================================================================
_ID_KEYS = ("id", "doc_id", "docid", "document_id", "case_id", "uid", "name", "file")
_TEXT_KEYS = ("text", "Text", "content", "body", "document", "judgment", "raw_text")
_START_KEYS = ("start", "start_offset", "start_char", "begin", "char_start", "startOffset",
               "start_idx")
_END_KEYS = ("end", "end_offset", "end_char", "stop", "char_end", "endOffset", "end_idx")
_LABEL_KEYS = ("label", "labels", "type", "entity_type", "tag", "entity_label", "category",
               "ner_label", "entity_group")
_ETEXT_KEYS = ("text", "entity_text", "entity", "span", "mention", "word", "surface")
_LIST_KEYS = ("entities", "annotations", "spans", "ner", "labels", "result", "gold", "entity",
              "ents", "mentions")


def load_texts(path):
    """{doc_id(str): text(str)} from {id: text}, {id: {"text":..}} or [{"id":..,"text":..}]."""
    obj = read_json(path)
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, str):
                out[str(k)] = v
            elif isinstance(v, dict):
                t = next((v[t] for t in _TEXT_KEYS if isinstance(v.get(t), str)), None)
                if t is None and isinstance(v.get("data"), dict):
                    t = next((v["data"][t] for t in _TEXT_KEYS
                              if isinstance(v["data"].get(t), str)), None)
                if t is None:
                    raise ValueError(f"{path}: cannot find the text of document {k!r}")
                out[str(k)] = t
            else:
                raise ValueError(f"{path}: unexpected value type {type(v)} for {k!r}")
    elif isinstance(obj, list):
        for i, rec in enumerate(obj):
            if not isinstance(rec, dict):
                raise ValueError(f"{path}: list element {i} is not an object")
            src = rec.get("data") if isinstance(rec.get("data"), dict) else rec
            did = next((rec[k] for k in _ID_KEYS if k in rec), None)
            t = next((src[k] for k in _TEXT_KEYS if isinstance(src.get(k), str)), None)
            if did is None or t is None:
                raise ValueError(f"{path}: element {i} has no id/text field: {list(rec)[:8]}")
            out[str(did)] = t
    else:
        raise ValueError(f"{path}: top level must be an object or a list")
    return out


def load_labels(path):
    labs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            for part in s.replace(",", " ").split():
                if part[:2] in ("B-", "I-", "E-", "S-"):
                    part = part[2:]
                if part != "O" and part not in labs:
                    labs.append(part)
    return labs


def _ann_from_seq(a):
    """[s, e, label(, text)] or [label, s, e(, text)] -> (s, e, label, text|None)."""
    if len(a) >= 3 and isinstance(a[0], int) and isinstance(a[1], int) and isinstance(a[2], str):
        return int(a[0]), int(a[1]), a[2], (a[3] if len(a) > 3 and isinstance(a[3], str) else None)
    if len(a) >= 3 and isinstance(a[0], str) and isinstance(a[1], int) and isinstance(a[2], int):
        return int(a[1]), int(a[2]), a[0], (a[3] if len(a) > 3 and isinstance(a[3], str) else None)
    return None


def _ann_from_dict(d):
    if isinstance(d.get("value"), dict):                     # Label-Studio style result
        merged = dict(d)
        merged.update(d["value"])
        d = merged
    s = next((d[k] for k in _START_KEYS if isinstance(d.get(k), int)), None)
    e = next((d[k] for k in _END_KEYS if isinstance(d.get(k), int)), None)
    lab = None
    for k in _LABEL_KEYS:
        v = d.get(k)
        if isinstance(v, str):
            lab = v
            break
        if isinstance(v, list) and v and isinstance(v[0], str):
            lab = v[0]
            break
    if s is None or e is None or lab is None:
        return None
    txt = next((d[k] for k in _ETEXT_KEYS if isinstance(d.get(k), str)), None)
    return int(s), int(e), lab, txt


def _parse_ann_list(v):
    """Parse whatever holds one document's annotations -> list of tuples (or None)."""
    if isinstance(v, dict):
        for k in _LIST_KEYS:
            if k in v and isinstance(v[k], list):
                r = _parse_ann_list(v[k])
                if r is not None:
                    return r
        return None
    if not isinstance(v, list):
        return None
    out = []
    for a in v:
        if isinstance(a, (list, tuple)):
            r = _ann_from_seq(a)
        elif isinstance(a, dict):
            if "result" in a and isinstance(a["result"], list):      # Label-Studio annotation
                sub = _parse_ann_list(a["result"])
                if sub is None:
                    return None
                out.extend(sub)
                continue
            r = _ann_from_dict(a)
        else:
            r = None
        if r is None:
            return None
        out.append(r)
    return out


def parse_gold(path):
    """{doc_id: [(start, end, label, text|None)]} + a human-readable schema description."""
    obj = read_json(path)
    gold = {}
    example = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            anns = _parse_ann_list(v)
            if anns is None:
                raise GateFailure(f"{path}: cannot parse the annotations of document {k!r}; "
                                  f"value starts with {json.dumps(v)[:300]}")
            gold[str(k)] = anns
            if example is None and anns:
                example = json.dumps(v if not isinstance(v, list) else v[:1])[:300]
        schema = "object {doc_id: annotations}"
    elif isinstance(obj, list):
        for i, rec in enumerate(obj):
            if not isinstance(rec, dict):
                raise GateFailure(f"{path}: list element {i} is not an object")
            did = next((rec[k] for k in _ID_KEYS if k in rec), None)
            if did is None:
                raise GateFailure(f"{path}: element {i} has no document id field")
            anns = _parse_ann_list({k: v for k, v in rec.items() if k not in _ID_KEYS})
            if anns is None:
                raise GateFailure(f"{path}: cannot parse annotations of element {i}: "
                                  f"{json.dumps(rec)[:300]}")
            gold[str(did)] = gold.get(str(did), []) + anns
            if example is None and anns:
                example = json.dumps(rec)[:300]
        schema = "list of {id, annotations} records"
    else:
        raise GateFailure(f"{path}: top level must be an object or a list")
    return gold, {"layout": schema, "example": example}


# =============================================================================================
# 3. gate G0 + overlap resolution
# =============================================================================================
def _ws_norm(s):
    return " ".join(s.split())


def check_g0(texts, gold, labels, allow_inclusive_ends=False, log=print):
    """Verify offsets against entity texts. Returns (clean_gold, report). Raises GateFailure."""
    rep = Counter()
    bad_examples, clean = [], {}
    unknown_labels = Counter()
    for did, anns in gold.items():
        if did not in texts:
            raise GateFailure(f"gold document {did!r} has no text")
        t = texts[did]
        out = []
        for (s, e, lab, et) in anns:
            rep["total"] += 1
            if lab not in labels:
                unknown_labels[lab] += 1
                continue
            if not (0 <= s < e <= len(t)):
                rep["out_of_bounds"] += 1
                bad_examples.append((did, s, e, lab, et, "out of bounds"))
                continue
            if et is None:
                rep["no_text"] += 1
            elif t[s:e] == et:
                rep["exact"] += 1
            elif t[s:e + 1] == et:
                rep["inclusive_end"] += 1
                if allow_inclusive_ends:
                    e = e + 1
                else:
                    bad_examples.append((did, s, e, lab, et, repr(t[s:e])))
                    continue
            elif _ws_norm(t[s:e]) == _ws_norm(et):
                rep["ws_normalised_match"] += 1
            else:
                rep["mismatch"] += 1
                bad_examples.append((did, s, e, lab, et, repr(t[s:e])))
                continue
            out.append((s, e, lab))
        clean[did] = out
    report = dict(rep)
    report["unknown_labels"] = dict(unknown_labels)
    report["examples_of_problems"] = [list(x) for x in bad_examples[:15]]
    if unknown_labels:
        raise GateFailure(f"G0: gold uses labels not in labels.txt: {dict(unknown_labels)}")
    if rep["inclusive_end"] and not allow_inclusive_ends:
        raise GateFailure(
            f"G0: {rep['inclusive_end']} entities only match with an INCLUSIVE end offset "
            f"(text[s:e+1]). The brief assumes exclusive ends; confirm the convention with the "
            f"official evaluator, then rerun with --allow-inclusive-ends. Examples: "
            f"{bad_examples[:3]}")
    if rep["mismatch"] or rep["out_of_bounds"]:
        raise GateFailure(f"G0 failed: {rep['mismatch']} mismatches, {rep['out_of_bounds']} "
                          f"out-of-bounds. Examples: {bad_examples[:5]}")
    log(f"   G0 passed: {rep['total']} entities | exact {rep['exact']} | "
        f"ws-normalised {rep['ws_normalised_match']} | without entity text {rep['no_text']}")
    return clean, report


def resolve_overlaps(anns):
    """Keep the longer (outer) entity when two overlap. Returns (kept_sorted, n_nested, n_cross)."""
    order = sorted(set(anns), key=lambda a: (-(a[1] - a[0]), a[0], a[2]))
    kept, nested, cross = [], 0, 0
    for a in order:
        clash = None
        for k in kept:
            if a[0] < k[1] and k[0] < a[1]:
                clash = k
                break
        if clash is None:
            kept.append(a)
        elif clash[0] <= a[0] and a[1] <= clash[1]:
            nested += 1
        else:
            cross += 1
    kept.sort()
    return kept, nested, cross


# =============================================================================================
# 4. IOB2 tagging / decoding
# =============================================================================================
def make_tag_names(labels):
    """index 0 = O, then B-X = 1 + 2k, I-X = 2 + 2k for label k."""
    names = ["O"]
    for lab in labels:
        names += [f"B-{lab}", f"I-{lab}"]
    return names


def tag_doc(td, ents, n_labels_map):
    """Token IOB2 tags for one document.

    td   : ner_tok.TokenizedDoc
    ents : sorted, non-overlapping [(start, end, label)]
    n_labels_map : {label: k}
    Returns (tags int16 array, stats dict).
    """
    n = len(td)
    owner = np.full(n, -1, dtype=np.int64)              # entity index per token (-1 = none)
    es_, ee_ = td.eff_start, td.eff_end
    straddle = 0
    j = 0
    for i in range(n):
        ts, te = int(es_[i]), int(ee_[i])
        if ts < 0:
            continue
        while j < len(ents) and ents[j][1] <= ts:
            j += 1
        best, best_ov = -1, 0
        k = j
        while k < len(ents) and ents[k][0] < te:
            ov = min(te, ents[k][1]) - max(ts, ents[k][0])
            if ov > best_ov:
                best, best_ov = k, ov
            k += 1
        if best >= 0:
            if best_ov * 2 > (te - ts):                 # majority of the token inside the entity
                owner[i] = best
                if best_ov < te - ts:
                    straddle += 1
            else:
                straddle += 1
    # whitespace-only tokens strictly inside an entity join it
    nonws = np.nonzero(es_ >= 0)[0]
    for i in range(n):
        if es_[i] >= 0 or td.raw_start[i] < 0:
            continue
        p = np.searchsorted(nonws, i) - 1
        q = p + 1
        if p >= 0 and q < len(nonws):
            a, b = owner[nonws[p]], owner[nonws[q]]
            if a >= 0 and a == b:
                owner[i] = a
    # continuation-byte tokens (own no character) follow the previous token
    for i in range(1, n):
        if td.raw_start[i] < 0 and owner[i - 1] >= 0:
            owner[i] = owner[i - 1]
    tags = np.zeros(n, dtype=np.int16)
    seen = set()
    for i in range(n):
        o = owner[i]
        if o < 0:
            continue
        k = n_labels_map[ents[o][2]]
        if o in seen:
            tags[i] = 2 + 2 * k
        else:
            tags[i] = 1 + 2 * k
            seen.add(o)
    lost = len(ents) - len(seen)
    return tags, {"straddling_tokens": straddle, "entities_without_tokens": lost}


def decode_tags(tags, td, labels):
    """IOB2 tags -> [(start, end, label)] using effective (whitespace-trimmed) token spans.

    A stray I-X (no open X span) starts a new span, the usual lenient convention; constrained
    Viterbi never produces one.
    """
    spans = []
    cur = None                                          # [k, start, end]
    es_, ee_ = td.eff_start, td.eff_end

    def close():
        if cur is not None and cur[1] >= 0 and cur[2] > cur[1]:
            spans.append((cur[1], cur[2], labels[cur[0]]))

    for i in range(len(tags)):
        t = int(tags[i])
        if t <= 0:
            close()
            cur = None
            continue
        k, is_b = (t - 1) // 2, (t % 2 == 1)
        s, e = int(es_[i]), int(ee_[i])
        if is_b or cur is None or cur[0] != k:
            close()
            cur = [k, s, e]
        else:
            if s >= 0 and cur[1] < 0:
                cur[1] = s
            if e >= 0:
                cur[2] = e
    close()
    return spans


# =============================================================================================
# 5. learned boundary snapping (used only if it raises the train ceiling; gate G1)
# =============================================================================================
def _is_punct(c):
    return not c.isalnum() and not c.isspace()


class SnapRules:
    """Single-character boundary fixes learned from the gold round trip.

    rules[label] = {"end_trim": set, "end_extend": set, "start_trim": set, "start_extend": set}
    end_trim  c : drop a final character c       end_extend  c : include a following char c
    start_trim c: drop a leading character c     start_extend c: include a preceding char c
    A rule is kept only if, on training documents, it fixes more spans than it breaks.
    """
    KINDS = ("end_trim", "end_extend", "start_trim", "start_extend")

    def __init__(self, rules=None):
        self.rules = rules or {}

    def to_json(self):
        return {lab: {k: sorted(v) for k, v in r.items()} for lab, r in self.rules.items()}

    @classmethod
    def from_json(cls, obj):
        return cls({lab: {k: set(v) for k, v in r.items()} for lab, r in (obj or {}).items()})

    @staticmethod
    def _candidate(kind, s, e, text):
        if kind == "end_trim" and e - s > 1 and _is_punct(text[e - 1]):
            return text[e - 1], (s, e - 1)
        if kind == "end_extend" and e < len(text) and _is_punct(text[e]):
            return text[e], (s, e + 1)
        if kind == "start_trim" and e - s > 1 and _is_punct(text[s]):
            return text[s], (s + 1, e)
        if kind == "start_extend" and s > 0 and _is_punct(text[s - 1]):
            return text[s - 1], (s - 1, e)
        return None, None

    def fit(self, decoded, gold, texts):
        """decoded/gold: {doc: [(s, e, label)]}."""
        self.rules = {}
        for kind in self.KINDS:
            score = defaultdict(lambda: [0, 0])          # (label, char) -> [fixes, breaks]
            for did, spans in decoded.items():
                gset = set(gold.get(did, []))
                text = texts[did]
                for (s, e, lab) in spans:
                    c, new = self._candidate(kind, s, e, text)
                    if c is None:
                        continue
                    now_ok = (s, e, lab) in gset
                    new_ok = (new[0], new[1], lab) in gset
                    if new_ok and not now_ok:
                        score[(lab, c)][0] += 1
                    elif now_ok and not new_ok:
                        score[(lab, c)][1] += 1
            for (lab, c), (fix, brk) in score.items():
                if fix > brk:
                    self.rules.setdefault(lab, {k: set() for k in self.KINDS})[kind].add(c)
            # apply this kind before learning the next one, mirroring apply()
            decoded = {d: [self._apply_kind(kind, sp, texts[d]) for sp in v]
                       for d, v in decoded.items()}
        return self

    def _apply_kind(self, kind, span, text):
        s, e, lab = span
        r = self.rules.get(lab)
        if not r or not r.get(kind):
            return span
        c, new = self._candidate(kind, s, e, text)
        if c is not None and c in r[kind]:
            return (new[0], new[1], lab)
        return span

    def apply(self, spans, text):
        out = []
        for sp in spans:
            for kind in self.KINDS:
                sp = self._apply_kind(kind, sp, text)
            out.append(sp)
        return out


def strict_f1(pred, gold):
    """Micro strict F1 over {doc: [(s,e,label)]} (used for the ceiling)."""
    tp = fp = fn = 0
    for did in set(pred) | set(gold):
        p, g = set(pred.get(did, [])), set(gold.get(did, []))
        tp += len(p & g)
        fp += len(p - g)
        fn += len(g - p)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return (2 * prec * rec / (prec + rec)) if prec + rec else 0.0


# =============================================================================================
# 6. chunking + merging (gate G2)
# =============================================================================================
def make_windows(n_tokens, chunk_len, stride):
    """Token windows [a, b) of at most chunk_len-1 tokens (one slot is the EOS prefix)."""
    w = chunk_len - 1
    assert w >= 1 and 1 <= stride <= w, "need 1 <= stride <= chunk_len - 1"
    if n_tokens <= w:
        return [(0, n_tokens)]
    out, a = [], 0
    while True:
        b = min(a + w, n_tokens)
        out.append((a, b))
        if b >= n_tokens:
            break
        a += stride
    return out


def centre_weights(length):
    """Weight of each position in a window: distance to the nearer edge (+1), so tokens near
    the centre of a chunk dominate where chunks overlap."""
    i = np.arange(length, dtype=np.float32)
    return np.minimum(i + 1, length - i)


def merge_window_scores(n_tokens, windows, window_scores):
    """Weighted average of per-window scores (each (len, C)) onto the document (n, C)."""
    C = window_scores[0].shape[1]
    acc = np.zeros((n_tokens, C), dtype=np.float64)
    wsum = np.zeros((n_tokens, 1), dtype=np.float64)
    for (a, b), sc in zip(windows, window_scores):
        w = centre_weights(b - a)[:, None]
        acc[a:b] += sc * w
        wsum[a:b] += w
    assert (wsum > 0).all(), "some tokens are not covered by any window"
    return (acc / wsum).astype(np.float32)


def build_train_chunks(docs, chunk_len, stride, neg_keep, seed, eos_id=0):
    """Training chunks: [EOS] + tokens, labels -100 on EOS. Keeps every chunk with an entity
    and a seeded fraction `neg_keep` of entity-free chunks."""
    rng = random.Random(seed)
    xs, ys = [], []
    n_pos = n_neg_total = n_neg_kept = 0
    for d in docs:
        ids, tags = d["ids"], d["tags"]
        for (a, b) in make_windows(len(ids), chunk_len, stride):
            y = tags[a:b]
            if (y > 0).any():
                n_pos += 1
            else:
                n_neg_total += 1
                if rng.random() >= neg_keep:
                    continue
                n_neg_kept += 1
            xs.append(np.concatenate([[eos_id], ids[a:b]]).astype(np.int32))
            ys.append(np.concatenate([[IGNORE], y]).astype(np.int16))
    return xs, ys, {"chunks": len(xs), "with_entities": n_pos, "empty_total": n_neg_total,
                    "empty_kept": n_neg_kept}


# =============================================================================================
# 7. dev split
# =============================================================================================
def make_dev_split(doc_ids, gold, labels, frac=0.15, seed=42, tries=2000):
    """Document-level split with every label present in dev (and in train)."""
    ids = sorted(doc_ids)
    n_dev = max(1, int(round(frac * len(ids))))
    labs_of = {d: {a[2] for a in gold.get(d, [])} for d in ids}
    rng = random.Random(seed)
    best, best_cov = None, -1
    for _ in range(tries):
        perm = ids[:]
        rng.shuffle(perm)
        dev, train = sorted(perm[:n_dev]), sorted(perm[n_dev:])
        dev_l = set().union(*(labs_of[d] for d in dev)) if dev else set()
        tr_l = set().union(*(labs_of[d] for d in train)) if train else set()
        cov = len(dev_l & set(labels)) + len(tr_l & set(labels))
        if cov > best_cov:
            best, best_cov = (train, dev), cov
        if cov == 2 * len(labels):
            break
    train, dev = best
    dev_l = set().union(*(labs_of[d] for d in dev))
    return {"train": train, "dev": dev, "seed": seed, "frac": frac,
            "labels_missing_in_dev": sorted(set(labels) - dev_l)}


# =============================================================================================
# 8. cache build (S0 + S1 + S2)
# =============================================================================================
def load_tokenizer(roots=None, explicit=None, log=print):
    """Find bpe_model.json (from the Phase-1 final/ dirs) and load it with the verbatim
    bpe_tokenizer.py. All copies found must be identical."""
    from bpe_tokenizer import BPETokenizer
    if explicit:
        paths = [explicit]
    else:
        paths = [p for p in find_files("bpe_model.json", roots or default_input_roots())
                 if "phase2_out" not in p]
    if not paths:
        raise FileNotFoundError("bpe_model.json not found; attach the Phase-1 output datasets")
    hashes = {file_md5(p) for p in paths}
    if len(hashes) != 1:
        raise GateFailure(f"found {len(hashes)} DIFFERENT bpe_model.json files: {paths}")
    tok = BPETokenizer.load(paths[0])
    return tok, paths[0], hashes.pop()


def build_cache(out_dir, roots=None, tokenizer_path=None, allow_inclusive_ends=False,
                max_overlap_frac=0.02, ceiling_min=0.97, dev_frac=0.15, seed=42,
                chunk_len=512, stride=256, log=print):
    from ner_tok import OffsetEncoder, verify_offsets
    t0 = time.time()
    os.makedirs(out_dir, exist_ok=True)
    report = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}

    # ---- S0: files, schema, stats ---------------------------------------------------------
    paths = locate_raw_files(os.path.join(out_dir, "raw"), roots, log=log)
    report["raw_files"] = {k: v for k, v in paths.items()}
    report["raw_md5"] = {k: file_md5(v) for k, v in paths.items()}
    labels = load_labels(paths["labels"])
    if sorted(labels) != sorted(EXPECTED_LABELS):
        log(f"   WARNING: labels.txt {labels} differs from the 12 expected labels")
    texts = {}
    fold_of = {}
    orig_id = {}                                          # cache key -> id as in the JSON file
    for fold in ("fold1", "fold2", "fold3"):
        t = load_texts(paths[f"{fold}_text"])
        for did, txt in t.items():
            key = did
            if key in texts:                              # id collision across folds
                key = f"{fold}:{did}"
            texts[key] = txt
            fold_of[key] = fold
            orig_id[key] = did
    gold_raw, schemas = {}, {}
    for fold in ("fold1", "fold2"):
        g, sch = parse_gold(paths[f"{fold}_gold"])
        schemas[fold] = sch
        ids_this_fold = {k for k, f in fold_of.items() if f == fold}
        for did, anns in g.items():
            key = did if did in ids_this_fold else f"{fold}:{did}"
            if key not in ids_this_fold:
                raise GateFailure(f"{fold}-gold has document {did!r} that is not in {fold}-text")
            gold_raw[key] = anns
        for did in ids_this_fold:
            gold_raw.setdefault(did, [])
    report["gold_schema"] = schemas
    log(f"   gold schema: {json.dumps(schemas)[:600]}")

    # ---- G0 ---------------------------------------------------------------------------------
    gold, g0 = check_g0(texts, gold_raw, labels, allow_inclusive_ends, log)
    report["G0"] = g0

    # ---- overlaps ---------------------------------------------------------------------------
    n_nested = n_cross = n_total = 0
    for did in gold:
        kept, ne, cr = resolve_overlaps(gold[did])
        n_total += len(set(gold[did]))
        n_nested += ne
        n_cross += cr
        gold[did] = kept
    frac = (n_nested + n_cross) / max(1, n_total)
    report["overlaps"] = {"nested_dropped": n_nested, "crossing_dropped": n_cross,
                          "fraction": frac}
    log(f"   overlaps: {n_nested} nested + {n_cross} crossing dropped ({frac:.2%})")
    if frac > max_overlap_frac:
        raise GateFailure(f"{frac:.2%} of gold entities are nested/overlapping (> "
                          f"{max_overlap_frac:.0%}); IOB2 cannot represent them -- stop and ask")

    # ---- S1: tokenise with offsets, tag, ceiling ---------------------------------------------
    tok, tok_path, tok_md5 = load_tokenizer(roots, tokenizer_path, log)
    report["tokenizer"] = {"path": tok_path, "md5": tok_md5, "vocab": len(tok)}
    enc = OffsetEncoder(tok)
    lab2k = {l: k for k, l in enumerate(labels)}
    docs = {}
    tag_stats = Counter()
    bad_offsets = 0
    for did, text in texts.items():
        td = enc.encode(text, check=True)
        bad_offsets += verify_offsets(text, td)
        rec = {"doc_id": did, "orig_id": orig_id[did], "fold": fold_of[did], "text": text,
               **td.to_dict()}
        if did in gold:
            tags, st = tag_doc(td, gold[did], lab2k)
            rec["tags"] = tags
            rec["gold"] = gold[did]
            tag_stats.update(st)
        docs[did] = rec
    if bad_offsets:
        raise GateFailure(f"offset encoder inconsistency: {bad_offsets} problems")
    report["tagging"] = dict(tag_stats)

    from ner_tok import TokenizedDoc
    decoded = {d: decode_tags(r["tags"], TokenizedDoc.from_dict(r), labels)
               for d, r in docs.items() if "tags" in r}
    ceiling_raw = strict_f1(decoded, gold)
    snap = SnapRules().fit(decoded, gold, texts)
    snapped = {d: snap.apply(v, texts[d]) for d, v in decoded.items()}
    ceiling_snap = strict_f1(snapped, gold)
    use_snap = ceiling_snap > ceiling_raw
    ceiling = max(ceiling_raw, ceiling_snap)
    # mismatch categories for the report
    cats = Counter()
    examples = []
    final = snapped if use_snap else decoded
    for did, g in gold.items():
        p = set(final.get(did, []))
        for (s, e, lab) in g:
            if (s, e, lab) in p:
                continue
            ov = [q for q in p if q[0] < e and s < q[1] and q[2] == lab]
            txt = texts[did]
            if not ov:
                cats["no_tokens_for_entity"] += 1
            elif txt[s].isspace() or txt[e - 1].isspace():
                cats["gold_starts_or_ends_on_whitespace"] += 1
            elif any(q[0] != s for q in ov):
                cats["start_inside_token"] += 1
            else:
                cats["end_inside_token"] += 1
            if len(examples) < 20:
                examples.append({"doc": did, "gold": [s, e, lab], "gold_text": txt[s:e],
                                 "decoded": [list(q) + [txt[q[0]:q[1]]] for q in ov][:2]})
    report["G1"] = {"ceiling_raw": ceiling_raw, "ceiling_with_snapping": ceiling_snap,
                    "use_snapping": use_snap, "snap_rules": snap.to_json(),
                    "mismatch_categories": dict(cats), "examples": examples}
    log(f"   G1 ceiling: raw {ceiling_raw:.4f} | with snapping {ceiling_snap:.4f} "
        f"(snapping {'ON' if use_snap else 'off'}) | mismatches {dict(cats)}")
    if ceiling < ceiling_min:
        write_json(os.path.join(out_dir, "data_report.json"), report)
        raise GateFailure(f"G1 failed: ceiling {ceiling:.4f} < {ceiling_min}. See "
                          f"data_report.json (G1.examples) -- stop and ask")

    # ---- dev split ---------------------------------------------------------------------------
    labeled = sorted(d for d in docs if "tags" in docs[d])
    split = make_dev_split(labeled, gold, labels, dev_frac, seed)
    split["test"] = sorted(d for d in docs if fold_of[d] == "fold3")
    if split["labels_missing_in_dev"]:
        log(f"   WARNING: labels missing in dev: {split['labels_missing_in_dev']}")
    report["split"] = {k: (len(v) if isinstance(v, list) else v) for k, v in split.items()}

    # ---- G2: tags -> windows -> merge -> decode == G1 spans -----------------------------------
    n_tags = 1 + 2 * len(labels)
    g2_bad = 0
    for did in labeled:
        r = docs[did]
        td = TokenizedDoc.from_dict(r)
        wins = make_windows(len(td), chunk_len, stride)
        onehot = np.eye(n_tags, dtype=np.float32)[r["tags"].astype(np.int64)]
        merged = merge_window_scores(len(td), wins, [onehot[a:b] for a, b in wins])
        if decode_tags(merged.argmax(1), td, labels) != decoded[did]:
            g2_bad += 1
    report["G2"] = {"docs_checked": len(labeled), "docs_mismatched": g2_bad,
                    "chunk_len": chunk_len, "stride": stride}
    if g2_bad:
        raise GateFailure(f"G2 failed on {g2_bad} documents")
    log(f"   G2 passed on {len(labeled)} documents (chunk {chunk_len}, stride {stride})")

    # ---- stats + save -------------------------------------------------------------------------
    stats = {"docs": Counter(fold_of.values()),
             "chars": {f: int(sum(len(texts[d]) for d in texts if fold_of[d] == f))
                       for f in ("fold1", "fold2", "fold3")},
             "tokens": {f: int(sum(len(docs[d]["ids"]) for d in docs if fold_of[d] == f))
                        for f in ("fold1", "fold2", "fold3")},
             "max_doc_tokens": int(max(len(r["ids"]) for r in docs.values())),
             "entities_per_label": dict(Counter(a[2] for g in gold.values() for a in g))}
    ent_tok_len = []
    for did in labeled:
        run = 0
        for v in docs[did]["tags"].tolist() + [0]:
            if v > 0 and v % 2 == 1:                      # B- starts a new entity
                if run:
                    ent_tok_len.append(run)
                run = 1
            elif v > 0 and run:                           # I- continues it
                run += 1
            else:
                if run:
                    ent_tok_len.append(run)
                run = 0
    if ent_tok_len:
        q = np.percentile(ent_tok_len, [50, 90, 99, 100])
        stats["entity_token_length_p50_p90_p99_max"] = [float(x) for x in q]
    report["stats"] = stats
    log(f"   stats: {json.dumps(stats, default=str)[:800]}")

    with open(os.path.join(out_dir, "docs.pkl.tmp"), "wb") as f:
        pickle.dump(docs, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(os.path.join(out_dir, "docs.pkl.tmp"), os.path.join(out_dir, "docs.pkl"))
    meta = {"labels": labels, "tag_names": make_tag_names(labels), "split": split,
            "snap_rules": snap.to_json() if use_snap else {}, "tokenizer_md5": tok_md5,
            "tokenizer_path": tok_path, "eos_id": int(tok.eos_token_id),
            "pad_id": int(tok.pad_token_id), "vocab_size": len(tok),
            "raw_md5": report["raw_md5"], "built": time.strftime("%Y-%m-%d %H:%M:%S"),
            "version": 1}
    write_json(os.path.join(out_dir, "meta.json"), meta)
    report["seconds"] = round(time.time() - t0, 1)
    write_json(os.path.join(out_dir, "data_report.json"), report)
    log(f"   data cache written to {out_dir} in {report['seconds']} s")
    return meta


def load_cache(data_dir):
    meta = read_json(os.path.join(data_dir, "meta.json"))
    if meta is None:
        raise FileNotFoundError(f"no data cache in {data_dir}")
    with open(os.path.join(data_dir, "docs.pkl"), "rb") as f:
        docs = pickle.load(f)
    return meta, docs


def find_or_build_cache(data_dir, roots=None, log=print, **kw):
    """Reuse a cache in data_dir or under the input roots, else build it. A cache is reused only
    if it was built with the SAME tokenizer (bpe_model.json md5) as the attached Phase-1 models."""
    roots = roots or default_input_roots()
    try:
        _tok, _p, tok_md5 = load_tokenizer(roots)
    except FileNotFoundError:
        tok_md5 = None

    def ok(d):
        meta = read_json(os.path.join(d, "meta.json"))
        return (isinstance(meta, dict) and meta.get("version") == 1 and "tag_names" in meta
                and os.path.exists(os.path.join(d, "docs.pkl"))
                and (tok_md5 is None or meta.get("tokenizer_md5") == tok_md5))
    if ok(data_dir):
        return data_dir
    for m in find_files("meta.json", roots):
        d = os.path.dirname(m)
        if ok(d):
            log(f"   using the data cache attached at {d}")
            return d
    build_cache(data_dir, roots=roots, log=log, **kw)
    return data_dir


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["build"])
    ap.add_argument("--out", default="/kaggle/working/phase2_data")
    ap.add_argument("--tokenizer", default=None, help="explicit path to bpe_model.json")
    ap.add_argument("--allow-inclusive-ends", action="store_true")
    ap.add_argument("--chunk-len", type=int, default=512)
    ap.add_argument("--stride", type=int, default=256)
    a = ap.parse_args()
    try:
        build_cache(a.out, tokenizer_path=a.tokenizer, allow_inclusive_ends=a.allow_inclusive_ends,
                    chunk_len=a.chunk_len, stride=a.stride)
    except GateFailure as e:
        print("\nSTOP AND ASK:", e, file=sys.stderr)
        sys.exit(2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/ner_eval.py
"""
ner_eval.py -- decoding + strict span metrics + error analysis (numpy only, no torch).

Tag layout (shared with ner_data.make_tag_names): 0 = O, B-X = 1 + 2k, I-X = 2 + 2k.

Decoding options (section 5.5 of the brief), all applied to document-level scores:
  * argmax
  * constrained Viterbi: only legal IOB2 transitions (O->{O,B-*}, B-X->{O,B-*,I-X},
    I-X->{O,B-*,I-X}; a sequence starts with O or B-*). For a linear head the scores are
    log-probabilities and the transition matrix is 0 / -inf; for a CRF head the learned
    transitions are added.
  * O-shift `delta`: subtract delta from the O score (delta > 0 -> more entities). Tuned on dev.

Metric (section 2): an entity counts only if (start, end, label) match exactly.
  per label P/R/F1, then two macro averages: over all labels, and over labels present in gold.
"""
import csv
import os
from collections import Counter, defaultdict

import numpy as np

NEG = -1e9


# =============================================================================================
# decoding
# =============================================================================================
def transition_constraints(n_labels):
    """allowed[prev, cur] and allowed_start[cur] for IOB2 with n_labels entity types."""
    C = 1 + 2 * n_labels
    allowed = np.ones((C, C), dtype=bool)
    for k in range(n_labels):
        i_tag = 2 + 2 * k
        allowed[:, i_tag] = False
        allowed[1 + 2 * k, i_tag] = True               # B-X -> I-X
        allowed[i_tag, i_tag] = True                   # I-X -> I-X
    start = np.ones(C, dtype=bool)
    start[2::2] = False                                # cannot start with I-*
    return allowed, start


def viterbi(scores, allowed, allowed_start, trans=None, start_scores=None, end_scores=None):
    """Best tag path for one document. scores: (T, C) float. Returns int64 (T,)."""
    T, C = scores.shape
    if T == 0:
        return np.zeros(0, dtype=np.int64)
    tr = np.where(allowed, 0.0 if trans is None else trans, NEG)
    s0 = np.where(allowed_start, 0.0, NEG)
    if start_scores is not None:
        s0 = s0 + start_scores
    score = s0 + scores[0]
    back = np.zeros((T, C), dtype=np.int32)
    for t in range(1, T):
        cand = score[:, None] + tr                     # (prev, cur)
        back[t] = cand.argmax(0)
        score = cand.max(0) + scores[t]
    if end_scores is not None:
        score = score + end_scores
    path = np.zeros(T, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def decode_scores(scores, td, labels, mode="viterbi", o_shift=0.0, is_logprob=True,
                  crf=None, snap=None, text=None):
    """Document scores (T, C) -> [(start, end, label)].

    is_logprob : True for averaged probabilities converted to log (linear head);
                 crf : dict with numpy "transitions", "start", "end" (CRF head) or None.
    """
    from ner_data import decode_tags
    sc = np.array(scores, dtype=np.float64, copy=True)
    if o_shift:
        sc[:, 0] -= o_shift
    if mode == "argmax":
        tags = sc.argmax(1)
    else:
        allowed, a_start = transition_constraints(len(labels))
        if crf is not None:
            tags = viterbi(sc, allowed, a_start, crf["transitions"], crf["start"], crf["end"])
        else:
            tags = viterbi(sc, allowed, a_start)
    spans = decode_tags(tags, td, labels)
    if snap is not None and text is not None:
        spans = snap.apply(spans, text)
    return spans


# =============================================================================================
# metrics
# =============================================================================================
def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def strict_scores(pred, gold, labels):
    """pred/gold: {doc_id: [(start, end, label)]} -> metrics dict."""
    cnt = {l: Counter() for l in labels}
    for did in set(gold) | set(pred):
        p = {tuple(x[:3]) for x in pred.get(did, [])}
        g = {tuple(x[:3]) for x in gold.get(did, [])}
        for x in p & g:
            cnt[x[2]]["tp"] += 1
        for x in p - g:
            if x[2] in cnt:
                cnt[x[2]]["fp"] += 1
        for x in g - p:
            cnt[x[2]]["fn"] += 1
    per = {}
    for l in labels:
        c = cnt[l]
        p, r, f = _prf(c["tp"], c["fp"], c["fn"])
        per[l] = {"precision": p, "recall": r, "f1": f, "tp": c["tp"], "fp": c["fp"],
                  "fn": c["fn"], "support": c["tp"] + c["fn"]}
    present = [l for l in labels if per[l]["support"] > 0]

    def macro(ls):
        if not ls:
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
        return {k: float(np.mean([per[l][k] for l in ls])) for k in ("precision", "recall", "f1")}

    tp = sum(c["tp"] for c in cnt.values())
    fp = sum(c["fp"] for c in cnt.values())
    fn = sum(c["fn"] for c in cnt.values())
    mp, mr, mf = _prf(tp, fp, fn)
    return {"macro_all": macro(labels), "macro_present": macro(present),
            "micro": {"precision": mp, "recall": mr, "f1": mf}, "per_label": per,
            "n_gold": tp + fn, "n_pred": tp + fp}


def headline(m):
    """The selection metric: strict macro-F1 over all labels."""
    return m["macro_all"]["f1"]


# =============================================================================================
# error analysis (section 10)
# =============================================================================================
def error_analysis(pred, gold, texts, labels, n_tokens=None):
    """Per-entity taxonomy, boundary offsets, span-level confusion, F1 by length bucket."""
    tax = Counter()
    d_start, d_end = Counter(), Counter()
    conf = defaultdict(Counter)                    # gold label -> predicted label ("O" = none)
    rows = []
    for did in sorted(set(gold) | set(pred)):
        g = sorted({tuple(x[:3]) for x in gold.get(did, [])})
        p = sorted({tuple(x[:3]) for x in pred.get(did, [])})
        pset = set(p)
        used = set()
        text = texts.get(did, "")
        for (s, e, l) in g:
            if (s, e, l) in pset:
                tax["exact"] += 1
                conf[l][l] += 1
                used.add((s, e, l))
                continue
            same_span = [q for q in p if q[0] == s and q[1] == e]
            ov_same = [q for q in p if q[0] < e and s < q[1] and q[2] == l]
            ov_any = [q for q in p if q[0] < e and s < q[1]]
            if same_span:
                kind = "right_span_wrong_label"
                q = same_span[0]
                conf[l][q[2]] += 1
            elif ov_same:
                kind = "right_label_wrong_boundary"
                q = min(ov_same, key=lambda q: abs(q[0] - s) + abs(q[1] - e))
                d_start[q[0] - s] += 1
                d_end[q[1] - e] += 1
                conf[l][l + "~boundary"] += 1
            elif ov_any:
                kind = "wrong_label_and_boundary"
                q = ov_any[0]
                conf[l][q[2] + "~boundary"] += 1
            else:
                kind = "missed"
                q = None
                conf[l]["O"] += 1
            tax[kind] += 1
            if q is not None:
                used.add(q)
            rows.append({"doc": did, "kind": kind, "gold_start": s, "gold_end": e,
                         "gold_label": l, "gold_text": text[s:e],
                         "pred_start": q[0] if q else "", "pred_end": q[1] if q else "",
                         "pred_label": q[2] if q else "", "pred_text": text[q[0]:q[1]] if q else ""})
        for q in p:
            if q in used or q in set(g):
                continue
            if not any(q[0] < e and s < q[1] for (s, e, _l) in g):
                tax["spurious"] += 1
                conf["O"][q[2]] += 1
                rows.append({"doc": did, "kind": "spurious", "gold_start": "", "gold_end": "",
                             "gold_label": "", "gold_text": "", "pred_start": q[0],
                             "pred_end": q[1], "pred_label": q[2], "pred_text": text[q[0]:q[1]]})
    # F1 by document length bucket (characters), micro, because buckets hold few documents
    buckets = [(0, 20_000), (20_000, 50_000), (50_000, 100_000), (100_000, 10 ** 12)]
    by_len = {}
    for lo, hi in buckets:
        ds = [d for d in gold if lo <= len(texts.get(d, "")) < hi]
        if not ds:
            continue
        sc = strict_scores({d: pred.get(d, []) for d in ds}, {d: gold[d] for d in ds}, labels)
        by_len[f"{lo}-{hi if hi < 10 ** 12 else 'inf'}"] = {"docs": len(ds),
                                                             "micro_f1": sc["micro"]["f1"],
                                                             "macro_f1": sc["macro_all"]["f1"]}
    return {"taxonomy": dict(tax),
            "boundary_start_offsets": {str(k): v for k, v in sorted(d_start.items())},
            "boundary_end_offsets": {str(k): v for k, v in sorted(d_end.items())},
            "confusion": {k: dict(v) for k, v in conf.items()},
            "f1_by_doc_length_chars": by_len}, rows


def write_errors_csv(path, rows):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    cols = ["doc", "kind", "gold_start", "gold_end", "gold_label", "gold_text", "pred_start",
            "pred_end", "pred_label", "pred_text"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            w.writerow({k: (v.replace("\n", "\\n") if isinstance(v, str) else v)
                        for k, v in r.items()})


def tune_decoding(dev_scores, dev_docs, gold, labels, is_logprob=True, crf=None, snap=None,
                  shifts=(-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0)):
    """Pick (mode, o_shift) maximising dev strict macro-F1. Returns (best, table)."""
    from ner_tok import TokenizedDoc
    table = []
    tds = {d: TokenizedDoc.from_dict(dev_docs[d]) for d in dev_scores}
    for mode in ("argmax", "viterbi"):
        for delta in (shifts if mode == "viterbi" else (0.0,)):
            pred = {d: decode_scores(dev_scores[d], tds[d], labels, mode, delta, is_logprob,
                                     crf, snap, dev_docs[d]["text"]) for d in dev_scores}
            m = strict_scores(pred, gold, labels)
            table.append({"mode": mode, "o_shift": delta, "macro_f1": headline(m),
                          "micro_f1": m["micro"]["f1"], "macro_present_f1": m["macro_present"]["f1"]})
    best = max(table, key=lambda r: (r["macro_f1"], r["mode"] == "viterbi", -abs(r["o_shift"])))
    return best, table


In [ ]:
%%writefile /kaggle/working/phase2/ner_crf.py
"""
ner_crf.py -- linear-chain CRF in pure PyTorch (no pytorch-crf).

Used by the optional CRF head (extra X3). Scores are computed in fp32 whatever the autocast
dtype. Illegal IOB2 transitions get a large negative constant, so the CRF can never learn or
decode an O -> I-X jump.

Shapes: emissions (B, T, C), tags (B, T) int64, mask (B, T) bool where every row is a prefix of
True values (the caller drops the EOS position and right-pads).

Constraint note: transitions O -> I-X / B-X -> I-Y are illegal everywhere; the START of a window
is left free because windows cut through entities (see _start).
"""
import torch
import torch.nn as nn

from ner_eval import transition_constraints

_ILLEGAL = -1e4


class CRF(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        C = 1 + 2 * n_labels
        self.num_tags = C
        self.transitions = nn.Parameter(torch.zeros(C, C))       # [prev, cur]
        self.start_transitions = nn.Parameter(torch.zeros(C))
        self.end_transitions = nn.Parameter(torch.zeros(C))
        allowed, a_start = transition_constraints(n_labels)
        self.register_buffer("trans_pen", torch.where(torch.from_numpy(allowed),
                                                      torch.tensor(0.0), torch.tensor(_ILLEGAL)),
                             persistent=False)
        self.register_buffer("start_pen", torch.where(torch.from_numpy(a_start),
                                                      torch.tensor(0.0), torch.tensor(_ILLEGAL)),
                             persistent=False)

    def _trans(self):
        return self.transitions + self.trans_pen

    def _start(self):
        # NOT constrained: a training window often starts inside an entity, so its first gold
        # tag can legitimately be I-X. The "no I-X at the start" rule only holds at the start
        # of a DOCUMENT, and document-level decoding (ner_eval.viterbi) enforces it there.
        return self.start_transitions

    def neg_log_likelihood(self, emissions, tags, mask, reduction="token_mean"):
        """-log p(tags | emissions). reduction: token_mean (sum / #tokens) | sum | none."""
        em = emissions.float()
        mask = mask.bool()
        B, T, C = em.shape
        trans, start = self._trans(), self._start()
        tags = tags.clamp(min=0)
        # gold path score
        score = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, T):
            m = mask[:, t]
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            score = score + step * m
        lengths = mask.sum(1)
        last = tags.gather(1, (lengths - 1).clamp(min=0).unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last]
        # partition function (forward algorithm)
        alpha = start.unsqueeze(0) + em[:, 0]                       # (B, C)
        for t in range(1, T):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0), dim=1) + em[:, t]
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        log_z = torch.logsumexp(alpha + self.end_transitions.unsqueeze(0), dim=1)
        nll = log_z - score
        if reduction == "none":
            return nll
        if reduction == "sum":
            return nll.sum()
        return nll.sum() / lengths.sum().clamp(min=1)

    @torch.no_grad()
    def viterbi(self, emissions, mask):
        """Best paths as python lists (one per row, length = mask row sum)."""
        em = emissions.float()
        mask = mask.bool()
        B, T, C = em.shape
        trans = self._trans()
        score = self._start().unsqueeze(0) + em[:, 0]
        history = []
        for t in range(1, T):
            cand = score.unsqueeze(2) + trans.unsqueeze(0)          # (B, prev, cur)
            best, idx = cand.max(1)
            nxt = best + em[:, t]
            score = torch.where(mask[:, t].unsqueeze(1), nxt, score)
            history.append(idx)
        score = score + self.end_transitions.unsqueeze(0)
        lengths = mask.sum(1).tolist()
        out = []
        for b in range(B):
            L = lengths[b]
            if L == 0:
                out.append([])
                continue
            tag = int(score[b].argmax())
            path = [tag]
            for t in range(L - 1, 0, -1):
                tag = int(history[t - 1][b, tag])
                path.append(tag)
            out.append(path[::-1])
        return out

    def export(self):
        """numpy transition tensors for document-level Viterbi in ner_eval.decode_scores."""
        return {"transitions": self.transitions.detach().float().cpu().numpy(),
                "start": self.start_transitions.detach().float().cpu().numpy(),
                "end": self.end_transitions.detach().float().cpu().numpy()}


In [ ]:
%%writefile /kaggle/working/phase2/ner_models.py
"""
ner_models.py -- Phase-1 trunk wrappers + NER heads.

The Phase-1 models' forward() only returns LM logits and is always causal. For NER we need the
final hidden states, optionally with bidirectional attention. `Trunk` re-implements only the
tiny "embed -> blocks -> final norm" loop and REUSES the original submodule objects (no copies;
lm_head is dropped, which also avoids a (B, T, 32000) logits tensor).

Attention modes (section 5.4):
  causal : exactly the Phase-1 masks (verified equal to the original forward in tests)
  bidir  : M1/M2 key-padding mask only; M3 symmetric sliding window |q-k| < window;
           M4 ALiBi with symmetric distance -slope*|q-k|. Padding is on the right.
"""
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_common import MODELS
from phase1_models import (MODEL_CLASSES, RMSNorm, alibi_bias, build_attn_mask, causal_allowed,
                           positions_from_mask)

MAX_POSITIONS = 1024        # every Phase-1 model was trained on 1024-token blocks


def torch_load(path, map_location="cpu"):
    """torch.load that works on old and new torch (weights_only appeared in 1.13)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        # weights_only=True refuses non-tensor pickles (our resumable checkpoints)
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)


def load_phase1_lm(phase1_dir):
    """Rebuild a Phase-1 model from final/{config.json, model.pt}; strict key match."""
    with open(os.path.join(phase1_dir, "config.json")) as f:
        cfg = json.load(f)
    cls, cfg_cls = MODEL_CLASSES[cfg["model_class"]]
    lm = cls(cfg_cls(**cfg["config"]))
    sd = torch_load(os.path.join(phase1_dir, "model.pt"))
    lm.load_state_dict(sd, strict=True)
    return lm, cfg


def kind_of(lm):
    name = type(lm).__name__
    return {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[name]


class Trunk(nn.Module):
    """embed -> selected blocks -> final norm, reusing the Phase-1 submodules."""

    def __init__(self, lm, attn_mode="bidir", layer_ids=None):
        super().__init__()
        assert attn_mode in ("causal", "bidir")
        self.kind = kind_of(lm)
        self.attn_mode = attn_mode
        c = lm.config
        self.window = None
        if self.kind == "gpt2":
            self.embed, self.wpe, blocks, self.norm = lm.wte, lm.wpe, lm.blocks, lm.ln_f
            self.n_positions = c.n_positions
            self.hidden_size = c.n_embd
        elif self.kind == "llama":
            self.embed, self.rotary, blocks, self.norm = lm.embed_tokens, lm.rotary, lm.layers, lm.norm
            self.window = c.sliding_window
            self.hidden_size = c.hidden_size
        else:
            self.embed, blocks, self.norm = lm.word_embeddings, lm.blocks, lm.ln_f
            self.register_buffer("slopes", lm.slopes.detach().clone(), persistent=False)
            self.hidden_size = c.hidden_size
        self.n_layers_original = len(blocks)
        self.layer_ids = list(range(len(blocks))) if layer_ids is None else list(layer_ids)
        assert all(0 <= i < len(blocks) for i in self.layer_ids)
        self.blocks = nn.ModuleList([blocks[i] for i in self.layer_ids])

    # ---------------------------------------------------------------- masks
    def _sdpa_mask(self, T, attention_mask, dev):
        """(mask, is_causal) for M1/M2/M3 blocks."""
        if self.attn_mode == "causal":
            return build_attn_mask(T, 0, attention_mask, self.window, dev)
        no_pad = bool(attention_mask.all())
        win = self.window if (self.window is not None and self.window < T) else None
        if no_pad and win is None:
            return None, False
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = torch.ones(1, 1, T, T, dtype=torch.bool, device=dev)
        if win is not None:
            allowed = allowed & ((q - k).abs() < win)[None, None]
        if not no_pad:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            allowed = allowed | (q == k)[None, None]          # no all-masked rows (pad queries)
        B = attention_mask.size(0)
        return allowed.expand(B, 1, T, T).contiguous(), False

    def _alibi(self, T, attention_mask, dev):
        pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
        if self.attn_mode == "causal":                          # identical to BloomStyleLM.forward
            allowed = causal_allowed(T, 0, attention_mask, None, dev)
            return alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)
        dist = (pos_k[:, None, :, None] - pos_k[:, None, None, :]).abs().float()
        bias = -self.slopes.view(1, -1, 1, 1) * dist
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = attention_mask[:, None, None, :].bool() | (q == k)[None, None]
        return bias.masked_fill(~allowed, float("-inf"))

    # ---------------------------------------------------------------- forward
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= MAX_POSITIONS, f"sequence of {T} tokens > {MAX_POSITIONS} (Phase-1 context)"
        dev = input_ids.device
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        pos = positions_from_mask(attention_mask, 0, T, B, dev)
        if self.kind == "gpt2":
            pos = pos.clamp(max=self.n_positions - 1)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids) + self.wpe(pos)
            for blk in self.blocks:
                x, _ = blk(x, mask, is_causal, None)
        elif self.kind == "llama":
            cos, sin = self.rotary(pos)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, cos, sin, mask, is_causal, None)
        else:
            bias = self._alibi(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, bias, None)
        return self.norm(x)

    def norm_modules(self):
        return [m for m in self.modules() if isinstance(m, (nn.LayerNorm, RMSNorm))]


class NERModel(nn.Module):
    """Trunk + dropout + Linear(H, C) [+ CRF]."""

    def __init__(self, trunk, n_labels, head="linear", dropout=0.1):
        super().__init__()
        self.trunk = trunk
        self.n_labels = n_labels
        self.num_tags = 1 + 2 * n_labels
        self.head_type = head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(trunk.hidden_size, self.num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.crf = None
        if head == "crf":
            from ner_crf import CRF
            self.crf = CRF(n_labels)

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None):
        h = self.trunk(input_ids, attention_mask)
        logits = self.classifier(self.dropout(h)).float()
        loss = None
        if labels is not None:
            if self.crf is None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
            else:                                   # drop the EOS position, mask = real labels
                lab = labels[:, 1:]
                loss = self.crf.neg_log_likelihood(logits[:, 1:], lab, lab != -100)
        return logits, loss


def build_ner_model(phase1_dir, n_labels, attn_mode="bidir", head="linear", layer_ids=None,
                    dropout=0.1):
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm                                          # unused blocks / lm_head are freed here
    return NERModel(trunk, n_labels, head, dropout), cfg


def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


# ---------------------------------------------------------------------------------------------
# trained-model reconstruction (teacher for E6, prediction, ensembles)
# ---------------------------------------------------------------------------------------------
def trainable_state_dict(model, half=False):
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    if half:
        sd = {n: (t.half() if t.is_floating_point() else t) for n, t in sd.items()}
    return sd


def load_trainable_state(model, sd):
    """Load a trainable-only state dict; every key must exist and match a trainable param."""
    params = dict(model.named_parameters())
    missing = [n for n in sd if n not in params]
    if missing:
        raise KeyError(f"checkpoint has unknown parameters: {missing[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            params[n].copy_(t.to(params[n].dtype))
    expected = {n for n, p in params.items() if p.requires_grad}
    absent = expected - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained_model(run_config, best_pt, phase1_dirs, device="cpu"):
    """Recreate the exact module structure of a finished run and load its best.pt."""
    from ner_peft import apply_technique
    mid = run_config["model"]
    p1 = phase1_dirs.get(mid) or run_config.get("phase1_dir")
    if not p1 or not os.path.exists(os.path.join(p1, "model.pt")):
        raise FileNotFoundError(f"Phase-1 weights for {mid} ({MODELS[mid]['arch']}) not found")
    model, _ = build_ner_model(p1, run_config["n_labels"], run_config["attn_mode"],
                               run_config["head"], run_config.get("layer_ids"))
    apply_technique(model, run_config["technique"], run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase2/ner_peft.py
"""
ner_peft.py -- fine-tuning / PEFT techniques, implemented from scratch (no `peft` library).

  full_ft      every trunk parameter + head            (E1; also KD students E6 / X2)
  lorma        LoRMA+  h = (I + s*B*A) W0 x  (+ bias)   (E2, compulsory)
  lora         LoRA    h = W0 x + s*B*A x   (+ bias)    (E3)
  adapters     Houlsby bottleneck after attention-out and MLP-out, + norms (E4)
  partial_ft   top-6 blocks + final norm + head, layer-wise LR decay (E5)
  bitfit       bias terms only (M1/M4)                  (extra X1)

`apply_technique(model, technique, cfg)` freezes everything, inserts the modules, unfreezes
exactly what the method trains, ASSERTS that the set of trainable parameter names equals the
expected set, and returns optimizer parameter groups (no weight decay on 1-D tensors).

LoRMA reference: Bihany, Patel & Modi (2025), "LoRMA: Low-Rank Multiplicative Adaptation for
LLMs", arXiv:2506.07621. We implement the additive-identity rank inflation (LoRMA+): the
frozen output y0 = W0 x is multiplied by (I + s*B*A) with B (d_out x r) = 0 at init, so the
layer starts as the exact frozen layer. The d_out x d_out product is never materialised:
y = y0 + s * (y0 @ A^T) @ B^T. For fused QKV projections each of q, k, v gets its own B, A.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from phase1_models import RMSNorm

# Linear layers adapted by LoRA / LoRMA, per architecture family (paths inside each block).
TARGETS = {
    "gpt2": ["attn.qkv", "attn.o_proj", "mlp.up_proj", "mlp.down_proj"],
    "llama": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj",
              "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    "bloom": ["self_attention.query_key_value", "self_attention.o_proj",
              "mlp.up_proj", "mlp.down_proj"],
}
FUSED_QKV = {"gpt2": "attn.qkv", "bloom": "self_attention.query_key_value"}
ADAPTER_SITES = {
    "gpt2": ["attn.o_proj", "mlp.down_proj"],
    "llama": ["attn.o_proj", "mlp.down_proj"],
    "bloom": ["self_attention.o_proj", "mlp.down_proj"],
}


def _get(mod, path):
    for p in path.split("."):
        mod = getattr(mod, p)
    return mod


def _set(mod, path, new):
    parts = path.split(".")
    for p in parts[:-1]:
        mod = getattr(mod, p)
    setattr(mod, parts[-1], new)


# =============================================================================================
# modules
# =============================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * B(A(dropout(x)))."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)),
                                         self.lora_B.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scale * (self.lora_B @ self.lora_A).to(self.base.weight.dtype)


class LoRMALinear(nn.Module):
    """LoRMA+ : y = (I + s*B*A) (W0 x) + b, applied separately to each output slice."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, slices=None):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.slices = slices or [(0, d_out)]
        assert self.slices[0][0] == 0 and self.slices[-1][1] == d_out
        assert all(a[1] == b[0] for a, b in zip(self.slices, self.slices[1:])), "slices must tile"
        self.lorma_A = nn.ParameterList()
        self.lorma_B = nn.ParameterList()
        for (s, e) in self.slices:
            A = nn.Parameter(torch.empty(r, e - s))
            nn.init.kaiming_uniform_(A, a=math.sqrt(5))
            self.lorma_A.append(A)
            self.lorma_B.append(nn.Parameter(torch.zeros(e - s, r)))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                  # frozen W0 x, bias added after
        parts = []
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            ys = y0[..., s:e]
            parts.append(ys + self.scale * ((ys @ A.to(ys.dtype).t()) @ B.to(ys.dtype).t()))
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):
        W0 = self.base.weight
        W = W0.clone()
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            W[s:e] = W0[s:e] + self.scale * (B @ (A @ W0[s:e]))
        return W


class Adapter(nn.Module):
    """Houlsby bottleneck: h + up(gelu(down(h))); up ~ N(0, 1e-3) so it starts near identity."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.normal_(self.up.weight, std=1e-3)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapped(nn.Module):
    """Wraps a sub-layer's output projection: adapter(base(x)), i.e. BEFORE the residual add."""

    def __init__(self, base: nn.Linear, bottleneck=64):
        super().__init__()
        self.base = base
        self.adapter = Adapter(base.out_features, bottleneck)

    def forward(self, x):
        return self.adapter(self.base(x))


# =============================================================================================
# applying a technique
# =============================================================================================
def _groups(named, lr, wd, tag):
    dec = [p for n, p in named if p.ndim >= 2]
    nodec = [p for n, p in named if p.ndim < 2]
    out = []
    if dec:
        out.append({"params": dec, "lr": lr, "weight_decay": wd, "name": f"{tag}/decay"})
    if nodec:
        out.append({"params": nodec, "lr": lr, "weight_decay": 0.0, "name": f"{tag}/no_decay"})
    return out


def apply_technique(model, technique, cfg, log=print):
    """Mutates `model` in place; returns optimizer param groups."""
    trunk = model.trunk
    kind = trunk.kind
    lr = float(cfg.get("lr", 1e-3))
    head_lr = float(cfg.get("head_lr", 1e-3))
    wd = float(cfg.get("weight_decay", 0.01))
    for p in model.parameters():
        p.requires_grad = False

    head_named = [(n, p) for n, p in model.named_parameters()
                  if n.startswith("classifier.") or n.startswith("crf.")]
    expected = {n for n, _ in head_named}
    groups = []

    if technique in ("full_ft", "kd", "student_hard"):
        named = [(n, p) for n, p in model.named_parameters() if n.startswith("trunk.")]
        expected |= {n for n, _ in named}
        for _, p in named:
            p.requires_grad = True
        groups += _groups(named, lr, wd, "trunk")

    elif technique in ("lora", "lorma"):
        r = int(cfg.get(f"{technique}_r", 16))
        alpha = float(cfg.get(f"{technique}_alpha", 32))
        for bi, blk in enumerate(trunk.blocks):
            for path in TARGETS[kind]:
                base = _get(blk, path)
                assert isinstance(base, nn.Linear), f"{path} is {type(base)}"
                if technique == "lora":
                    new = LoRALinear(base, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                else:
                    slices = None
                    if FUSED_QKV.get(kind) == path:
                        d = trunk.hidden_size
                        assert base.out_features == 3 * d
                        slices = [(0, d), (d, 2 * d), (2 * d, 3 * d)]
                    new = LoRMALinear(base, r, alpha, slices)
                _set(blk, path, new)
        named = [(n, p) for n, p in model.named_parameters()
                 if ".lora_A" in n or ".lora_B" in n or ".lorma_A" in n or ".lorma_B" in n]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, technique)

    elif technique == "adapters":
        bott = int(cfg.get("adapter_bottleneck", 64))
        for blk in trunk.blocks:
            for path in ADAPTER_SITES[kind]:
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
        named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
        norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
        named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "adapters")

    elif technique == "bitfit":
        named = [(n, p) for n, p in model.named_parameters()
                 if n.startswith("trunk.") and n.endswith(".bias")]
        if not named:
            raise ValueError(f"BitFit needs bias terms; the {kind} architecture has none")
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "bitfit")

    elif technique == "partial_ft":
        k = int(cfg.get("partial_top_k", 6))
        decay = float(cfg.get("layer_decay", 0.85))
        L = len(trunk.blocks)
        k = min(k, L)
        for i in range(L - k, L):
            named = [(f"trunk.blocks.{i}.{n}", p) for n, p in trunk.blocks[i].named_parameters()]
            for _, p in named:
                p.requires_grad = True
            expected |= {n for n, _ in named}
            groups += _groups(named, lr * decay ** (L - 1 - i), wd, f"block{i}")
        named = [(f"trunk.norm.{n}", p) for n, p in trunk.norm.named_parameters()]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "final_norm")
    else:
        raise ValueError(f"unknown technique {technique!r}")

    for _, p in head_named:
        p.requires_grad = True
    groups += _groups(head_named, head_lr, wd, "head")

    actual = {n for n, p in model.named_parameters() if p.requires_grad}
    if actual != expected:
        raise AssertionError(f"trainable-parameter mismatch: unexpected "
                             f"{sorted(actual - expected)[:5]}, missing {sorted(expected - actual)[:5]}")
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"   [{technique}] trainable {n_tr:,} / {n_all:,} params ({100 * n_tr / n_all:.3f}%) "
        f"in {len(actual)} tensors")
    return groups


def summarize_trainable(model, max_lines=40):
    rows = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    lines = [f"{n}  {k:,}" for n, k in rows[:max_lines]]
    if len(rows) > max_lines:
        lines.append(f"... and {len(rows) - max_lines} more tensors")
    return lines


In [ ]:
%%writefile /kaggle/working/phase2/ner_predict.py
"""
ner_predict.py -- inference, dev evaluation, test prediction, submission, DONE.json.

    python ner_predict.py --run-dir OUT/E2_lorma/M2_llamastyle            # after training
    python ner_predict.py --ensemble RUN1,RUN2,... --out OUT/X5_ensemble  # extra X5

Inference: every document is cut into overlapping windows ([EOS] + up to chunk_len-1 tokens,
stride `infer_stride`), windows are batched, and per-token scores are merged back with weights
that favour the centre of each window (ner_data.merge_window_scores). Linear head: averaged
softmax probabilities, decoded on their logs. CRF head: averaged emissions + learned transitions.
Decoding mode and O-shift are tuned on dev (ner_eval.tune_decoding).
"""
import argparse
import contextlib
import os
import shutil
import sys
import time
import zipfile

import numpy as np
import torch

from ner_common import find_phase1_models, read_json, write_json
from ner_data import IGNORE, SnapRules, load_cache, make_windows, merge_window_scores
from ner_eval import (decode_scores, error_analysis, headline, strict_scores, tune_decoding,
                      write_errors_csv)
from ner_tok import TokenizedDoc


def amp_setup(device):
    """The Phase-1 notebooks' AMP pattern (cell 21): bf16 on Ampere+, fp16 on T4/P100."""
    if device.type == "cuda" and torch.cuda.get_device_capability(device)[0] >= 8:
        amp_dtype = torch.bfloat16
    elif device.type == "cuda":
        amp_dtype = torch.float16
    else:
        amp_dtype = None

    def amp_ctx():
        if amp_dtype is None or device.type != "cuda":
            return contextlib.nullcontext()
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return amp_dtype, amp_ctx


# =============================================================================================
# window inference
# =============================================================================================
@torch.no_grad()
def infer_scores(model, docs, doc_ids, chunk_len, stride, device, amp_ctx, eos_id=0, pad_id=1,
                 batch_size=32):
    """{doc_id: (T, C) float32}. Linear head -> merged probabilities; CRF -> merged emissions."""
    was_training = model.training
    model.eval()
    items = []                                           # (doc_id, a, b)
    for d in doc_ids:
        for (a, b) in make_windows(len(docs[d]["ids"]), chunk_len, stride):
            items.append((d, a, b))
    order = sorted(range(len(items)), key=lambda i: items[i][2] - items[i][1])
    per_window = [None] * len(items)
    use_probs = getattr(model, "crf", None) is None
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        L = max(items[i][2] - items[i][1] for i in idx) + 1
        x = torch.full((len(idx), L), pad_id, dtype=torch.long)
        m = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            d, a, b = items[i]
            n = b - a
            x[r, 0] = eos_id
            x[r, 1:n + 1] = torch.from_numpy(docs[d]["ids"][a:b].astype(np.int64))
            m[r, :n + 1] = 1
        x, m = x.to(device), m.to(device)
        with amp_ctx():
            logits, _ = model(x, m)
        logits = logits.float()
        out = torch.softmax(logits, -1) if use_probs else logits
        out = out.cpu().numpy()
        for r, i in enumerate(idx):
            d, a, b = items[i]
            per_window[i] = out[r, 1:b - a + 1]
    model.train(was_training)
    by_doc = {}
    for i, (d, a, b) in enumerate(items):
        by_doc.setdefault(d, ([], []))
        by_doc[d][0].append((a, b))
        by_doc[d][1].append(per_window[i])
    return {d: merge_window_scores(len(docs[d]["ids"]), w, s) for d, (w, s) in by_doc.items()}


def to_decode_scores(scores, is_probs):
    if is_probs:
        return {d: np.log(np.clip(v, 1e-9, 1.0)) for d, v in scores.items()}
    return scores


def decode_all(dec_scores, docs, labels, mode, o_shift, crf=None, snap=None):
    return {d: decode_scores(v, TokenizedDoc.from_dict(docs[d]), labels, mode, o_shift,
                             crf=crf, snap=snap, text=docs[d]["text"]) for d, v in dec_scores.items()}


def quick_dev_f1(model, meta, docs, cfg, device, amp_ctx, snap=None):
    """Used during training: constrained Viterbi, no O-shift. Returns (macro_f1, metrics)."""
    dev_ids = meta["split"]["dev"]
    sc = infer_scores(model, docs, dev_ids, cfg["chunk_len"], cfg["infer_stride"], device, amp_ctx,
                      meta["eos_id"], meta["pad_id"], cfg.get("eval_batch", 32))
    crf = model.crf.export() if getattr(model, "crf", None) is not None else None
    pred = decode_all(to_decode_scores(sc, crf is None), docs, meta["labels"], "viterbi", 0.0,
                      crf, snap)
    gold = {d: docs[d]["gold"] for d in dev_ids}
    m = strict_scores(pred, gold, meta["labels"])
    return headline(m), m


# =============================================================================================
# submission
# =============================================================================================
def write_submission(sub_dir, preds_by_cache_id, docs, test_ids):
    os.makedirs(sub_dir, exist_ok=True)
    pred = {}
    debug = {}
    for d in test_ids:
        oid = docs[d]["orig_id"]
        spans = sorted(preds_by_cache_id.get(d, []))
        pred[oid] = [[int(s), int(e), l] for (s, e, l) in spans]
        debug[oid] = [[int(s), int(e), l, docs[d]["text"][s:e]] for (s, e, l) in spans]
    pj = os.path.join(sub_dir, "prediction.json")
    write_json(pj, pred, indent=None)
    write_json(os.path.join(sub_dir, "entities_debug.json"), debug)
    zp = os.path.join(sub_dir, "submission.zip")
    with zipfile.ZipFile(zp + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")               # at the ROOT of the archive
    os.replace(zp + ".tmp", zp)
    return zp


def _model_stats(model, device):
    n_all = sum(p.numel() for p in model.parameters())
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": n_all, "trainable_params": n_tr,
            "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated(device) / 1e9
                                if device.type == "cuda" else 0.0)}


def evaluate_and_predict(models, meta, docs, cfg, out_dir, device, log=print):
    """Shared by single runs and ensembles. `models` is a list (len 1 for a normal run)."""
    _, amp_ctx = amp_setup(device)
    labels = meta["labels"]
    snap = SnapRules.from_json(meta.get("snap_rules") or {})
    dev_ids, test_ids = meta["split"]["dev"], meta["split"]["test"]
    is_probs = all(getattr(m, "crf", None) is None for m in models)
    crf = None if is_probs else models[0].crf.export()
    if not is_probs and len(models) > 1:
        raise ValueError("ensembles are supported for linear heads only")

    def scores_for(ids):
        t0 = time.time()
        acc = None
        for m in models:
            sc = infer_scores(m, docs, ids, cfg["chunk_len"], cfg["infer_stride"], device,
                              amp_ctx, meta["eos_id"], meta["pad_id"], cfg.get("eval_batch", 32))
            acc = sc if acc is None else {d: acc[d] + sc[d] for d in sc}
        acc = {d: v / len(models) for d, v in acc.items()}
        n_tok = sum(len(docs[d]["ids"]) for d in ids)
        return to_decode_scores(acc, is_probs), n_tok / max(1e-9, time.time() - t0)

    # ---- dev: tune decoding, full report ----------------------------------------------------
    dev_sc, dev_tps = scores_for(dev_ids)
    gold = {d: docs[d]["gold"] for d in dev_ids}
    best, table = tune_decoding(dev_sc, {d: docs[d] for d in dev_ids}, gold, labels,
                                crf=crf, snap=snap)
    pred_dev = decode_all(dev_sc, docs, labels, best["mode"], best["o_shift"], crf, snap)
    metrics = strict_scores(pred_dev, gold, labels)
    texts = {d: docs[d]["text"] for d in dev_ids}
    analysis, rows = error_analysis(pred_dev, gold, texts, labels)
    # effect of decoding choices (report section)
    argmax_row = next(r for r in table if r["mode"] == "argmax")
    vit0 = next(r for r in table if r["mode"] == "viterbi" and r["o_shift"] == 0.0)
    dev_dir = os.path.join(out_dir, "dev")
    write_errors_csv(os.path.join(dev_dir, "errors.csv"), rows)
    report = {"decoding": best, "decoding_table": table,
              "argmax_vs_viterbi": {"argmax": argmax_row["macro_f1"], "viterbi_shift0": vit0["macro_f1"],
                                    "viterbi_tuned": best["macro_f1"]},
              "metrics": metrics, "analysis": analysis,
              "inference_tokens_per_s": dev_tps, **_model_stats(models[0], device)}
    write_json(os.path.join(dev_dir, "dev_report.json"), report)
    log(f"   dev strict macro-F1 {headline(metrics):.4f} (micro {metrics['micro']['f1']:.4f}) | "
        f"decode {best['mode']} shift {best['o_shift']} | argmax {argmax_row['macro_f1']:.4f}")

    # ---- test: fold3 ----------------------------------------------------------------------------
    test_sc, _ = scores_for(test_ids)
    pred_test = decode_all(test_sc, docs, labels, best["mode"], best["o_shift"], crf, snap)
    zp = write_submission(os.path.join(out_dir, "submission"), pred_test, docs, test_ids)
    from validate_submission import validate
    vr = validate(zp, {docs[d]["orig_id"]: docs[d]["text"] for d in test_ids}, labels)
    write_json(os.path.join(out_dir, "submission", "validation_report.json"), vr)
    if not vr["ok"]:
        raise RuntimeError(f"submission failed validation: {vr['errors'][:5]}")
    log(f"   submission written and validated: {zp} ({vr['n_spans']} spans)")
    return metrics, report


def run_single(run_dir, data_dir=None, device=None, log=print):
    from ner_models import rebuild_trained_model
    cfg = read_json(os.path.join(run_dir, "logs", "run_config.json"))
    data_dir = data_dir or cfg["data_dir"]
    meta, docs = load_cache(data_dir)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    best_pt = os.path.join(run_dir, "checkpoints", "best.pt")
    if not os.path.exists(best_pt):
        raise FileNotFoundError(f"{best_pt} missing -- training did not produce a best model")
    p1 = find_phase1_models()
    if cfg.get("phase1_dir"):
        p1.setdefault(cfg["model"], cfg["phase1_dir"])
    model = rebuild_trained_model(cfg, best_pt, p1, device)
    metrics, report = evaluate_and_predict([model], meta, docs, cfg, run_dir, device, log)
    train_sum = read_json(os.path.join(run_dir, "logs", "train_summary.json"), {})
    done = {"exp": cfg["exp"], "model": cfg["model"], "technique": cfg["technique"],
            "arch": cfg["arch"], "dev_macro_f1": headline(metrics),
            "dev_macro_precision": metrics["macro_all"]["precision"],
            "dev_macro_recall": metrics["macro_all"]["recall"],
            "dev_macro_present_f1": metrics["macro_present"]["f1"],
            "dev_micro_f1": metrics["micro"]["f1"], "decoding": report["decoding"],
            "trainable_params": report["trainable_params"], "total_params": report["total_params"],
            "inference_tokens_per_s": report["inference_tokens_per_s"],
            "train": train_sum, "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
    write_json(os.path.join(run_dir, "DONE.json"), done)
    return done


def run_ensemble(run_dirs, out_dir, device=None, log=print):
    from ner_models import rebuild_trained_model
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # each member is a run folder (logs/run_config.json, checkpoints/best.pt) or a BEST/ folder
    # (run_config.json, best.pt)
    cfgs = [read_json(os.path.join(r, "logs", "run_config.json")) or
            read_json(os.path.join(r, "run_config.json")) for r in run_dirs]
    missing = [r for r, c in zip(run_dirs, cfgs) if c is None]
    if missing:
        raise FileNotFoundError(f"no run_config.json in {missing}")
    data_dirs = {c["data_dir"] for c in cfgs}
    meta, docs = load_cache(sorted(data_dirs)[0])
    p1 = find_phase1_models()
    models = []
    for r, c in zip(run_dirs, cfgs):
        bp = os.path.join(r, "checkpoints", "best.pt")
        if not os.path.exists(bp):
            bp = os.path.join(r, "best.pt")                    # a BEST/ folder
        models.append(rebuild_trained_model(c, bp, p1, device))
    cfg = dict(cfgs[0])
    metrics, report = evaluate_and_predict(models, meta, docs, cfg, out_dir, device, log)
    write_json(os.path.join(out_dir, "DONE.json"),
               {"exp": "X5", "model": "+".join(c["model"] for c in cfgs),
                "technique": "ensemble:" + "+".join(c["technique"] for c in cfgs),
                "members": run_dirs, "dev_macro_f1": headline(metrics),
                "dev_micro_f1": metrics["micro"]["f1"], "decoding": report["decoding"]})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir")
    ap.add_argument("--data-dir", default=None)
    ap.add_argument("--ensemble", default=None, help="comma-separated run (or BEST) dirs")
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    if a.ensemble:
        run_ensemble([p for p in a.ensemble.split(",") if p], a.out)
    else:
        done = run_single(a.run_dir, a.data_dir)
        print(f"DONE {done['exp']} {done['model']}: dev macro-F1 {done['dev_macro_f1']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/ner_train.py
"""
ner_train.py -- one training run for one (experiment, model). Launched by run_experiment.py:

    python ner_train.py --run-dir RUN --mode train                       # 1 GPU / CPU
    python -m torch.distributed.run --nproc_per_node=2 ner_train.py --run-dir RUN --mode train

Reads RUN/logs/run_config.json. Modes:
  smoke    ~30 optimizer steps; writes logs/smoke.json (tokens/s, memory, projected minutes)
  overfit  64 entity chunks, 150 steps; must drive the loss down (catches label misalignment)
  probe    lr x {1/3, 1, 3} for a short budget each; writes logs/lr_probe.json
  train    the real run (checkpoints, per-epoch dev F1, early stopping, time guard, resume)

Exit codes: 0 ok | 2 time budget reached (checkpoint saved) | 3 out of memory in DDP |
            4 NaN abort | 5 early-fail watchdog | 1 any other error
"""
import argparse
import datetime
import json
import math
import os
import random
import sys
import time
import traceback

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F

from ner_common import read_json, write_json
from ner_data import IGNORE, SnapRules, build_train_chunks, load_cache
from ner_models import (build_ner_model, count_params, load_trainable_state,
                        rebuild_trained_model, torch_load, trainable_state_dict)
from ner_peft import apply_technique
from ner_predict import amp_setup, quick_dev_f1

EXIT_TIME, EXIT_OOM, EXIT_NAN, EXIT_EARLYFAIL = 2, 3, 4, 5
FULL_PARAM = ("full_ft", "kd", "student_hard", "partial_ft")


def is_oom(e):
    oom_cls = getattr(torch.cuda, "OutOfMemoryError", None)
    return (oom_cls is not None and isinstance(e, oom_cls)) or "out of memory" in str(e).lower()


class Ctx:
    """Distributed / device context."""

    def __init__(self):
        self.world = int(os.environ.get("WORLD_SIZE", "1"))
        self.rank = int(os.environ.get("RANK", "0"))
        self.local = int(os.environ.get("LOCAL_RANK", "0"))
        cuda = torch.cuda.is_available()
        if self.world > 1:
            if cuda:
                torch.cuda.set_device(self.local)
            dist.init_process_group("nccl" if cuda else "gloo",
                                    timeout=datetime.timedelta(minutes=10))
        self.device = torch.device("cuda", self.local) if cuda else torch.device("cpu")
        self.main = self.rank == 0

    def barrier(self):
        if self.world > 1:
            dist.barrier()

    def bcast(self, values):
        """Broadcast a list of floats from rank 0."""
        t = torch.tensor(values, dtype=torch.float64, device=self.device)
        if self.world > 1:
            dist.broadcast(t, 0)
        return t.tolist()

    def all_min(self, v):
        t = torch.tensor([float(v)], device=self.device)
        if self.world > 1:
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
        return t.item()

    def close(self):
        if self.world > 1 and dist.is_initialized():
            dist.destroy_process_group()


def log(ctx, run_dir, msg, record=None):
    if not ctx.main:
        return
    print(msg, flush=True)
    if record is not None:
        with open(os.path.join(run_dir, "logs", "train_log.jsonl"), "a") as f:
            f.write(json.dumps(record) + "\n")


# =============================================================================================
# data
# =============================================================================================
class EpochOrder:
    """Deterministic global permutation per epoch; rank r takes every world-th item after
    skipping the first `skip` (already consumed, used on resume)."""

    def __init__(self, n, seed):
        self.n, self.seed = n, seed

    def indices(self, epoch, rank, world, skip=0):
        perm = np.random.default_rng(self.seed + 1000 * epoch).permutation(self.n)
        rest = perm[skip:]
        if len(rest) == 0:
            return []
        # every rank must run the SAME number of micro-batches (or DDP deadlocks): pad by
        # wrapping around, exactly like torch's DistributedSampler
        per_rank = math.ceil(len(rest) / world)
        if per_rank * world > len(rest):
            rest = np.concatenate([rest, np.resize(perm, per_rank * world - len(rest))])
        return rest[rank::world].tolist()


def collate(xs, ys, idx, pad_id):
    L = max(len(xs[i]) for i in idx)
    x = torch.full((len(idx), L), pad_id, dtype=torch.long)
    y = torch.full((len(idx), L), IGNORE, dtype=torch.long)
    m = torch.zeros((len(idx), L), dtype=torch.long)
    for r, i in enumerate(idx):
        n = len(xs[i])
        x[r, :n] = torch.from_numpy(xs[i].astype(np.int64))
        y[r, :n] = torch.from_numpy(ys[i].astype(np.int64))
        m[r, :n] = 1
    return x, y, m


# =============================================================================================
# model / optimizer
# =============================================================================================
def make_model(cfg, ctx, lr_mult=1.0):
    mid = cfg["model"]
    model, _ = build_ner_model(cfg["phase1_dir"], cfg["n_labels"], cfg["attn_mode"], cfg["head"],
                               cfg.get("layer_ids"), cfg.get("dropout", 0.1))
    c = dict(cfg)
    c["lr"] = cfg["lr"] * lr_mult
    c["head_lr"] = cfg["head_lr"] * (lr_mult if cfg["technique"] in FULL_PARAM else 1.0)
    groups = apply_technique(model, cfg["technique"], c,
                             log=(lambda m: print(m, flush=True)) if ctx.main else (lambda m: None))
    model.to(ctx.device)
    for g in groups:
        g["initial_lr"] = g["lr"]
    opt = torch.optim.AdamW([{k: v for k, v in g.items() if k != "name"} for g in groups],
                            betas=(0.9, 0.98), eps=1e-8)
    return model, opt


def make_scheduler(opt, total_steps, warmup_frac):
    warm = max(1, int(round(warmup_frac * total_steps)))

    def f(step):
        if step < warm:
            return (step + 1) / warm
        return max(0.0, (total_steps - step) / max(1, total_steps - warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def wrap_ddp(model, ctx):
    if ctx.world == 1:
        return model
    from torch.nn.parallel import DistributedDataParallel as DDP
    kw = {"device_ids": [ctx.local]} if ctx.device.type == "cuda" else {}
    return DDP(model, find_unused_parameters=False, **kw)


def load_teacher(cfg, ctx):
    tdir = cfg["teacher_dir"]
    tcfg = read_json(os.path.join(tdir, "run_config.json"))
    from ner_common import find_phase1_models
    p1 = find_phase1_models()
    if tcfg.get("phase1_dir"):
        p1.setdefault(tcfg["model"], tcfg["phase1_dir"])
    teacher = rebuild_trained_model(tcfg, os.path.join(tdir, "best.pt"), p1, ctx.device)
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher.eval(), tcfg


def kd_loss(student_logits, teacher_logits, labels, alpha, T, class_weights=None):
    mask = labels != IGNORE
    s = student_logits[mask].float()
    t = teacher_logits[mask].float()
    hard = F.cross_entropy(s, labels[mask], weight=class_weights)
    soft = F.kl_div(F.log_softmax(s / T, -1), F.log_softmax(t / T, -1), log_target=True,
                    reduction="batchmean") * (T * T)
    return alpha * hard + (1 - alpha) * soft


def class_weight_tensor(cfg, ys, num_tags, device):
    if cfg.get("class_weight", "none") != "sqrt_inv":
        return None
    cnt = np.bincount(np.concatenate([y[y >= 0] for y in ys]).astype(np.int64), minlength=num_tags)
    w = 1.0 / np.sqrt(np.maximum(cnt, 1))
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)


# =============================================================================================
# the training loop (shared by every mode)
# =============================================================================================
def train_loop(cfg, ctx, run_dir, meta, docs, xs, ys, mode, lr_mult=1.0, max_steps=None,
               epochs=None, resume=True, eval_each_epoch=True):
    """Returns a summary dict. Raises SystemExit with the exit codes in the module doc."""
    torch.manual_seed(cfg["seed"] + ctx.rank)
    random.seed(cfg["seed"] + ctx.rank)
    np.random.seed(cfg["seed"] + ctx.rank)
    model, opt = make_model(cfg, ctx, lr_mult)
    raw_model = model
    amp_dtype, amp_ctx = amp_setup(ctx.device)
    scaler = make_scaler(amp_dtype == torch.float16)
    teacher = None
    if cfg["technique"] == "kd":
        teacher, _ = load_teacher(cfg, ctx)
    if cfg["technique"] in ("kd", "student_hard") and cfg.get("enforce_size", True):
        n_s = count_params(raw_model)
        assert 50e6 <= n_s <= 300e6, f"student has {n_s / 1e6:.1f}M params (must be 50M-300M)"
        if teacher is not None:
            assert n_s < count_params(teacher), "student must be smaller than the teacher"
    cw = class_weight_tensor(cfg, ys, raw_model.num_tags, ctx.device)
    snap = SnapRules.from_json(meta.get("snap_rules") or {})
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    half_best = cfg["technique"] in FULL_PARAM

    # ---- micro-batch: memory probe on the longest possible batch -----------------------------
    mb = int(cfg.get("micro_batch", 16))
    if ctx.device.type == "cuda":
        L = cfg["chunk_len"]
        while True:
            try:
                x = torch.randint(3, meta["vocab_size"], (mb, L), device=ctx.device)
                with amp_ctx():
                    logits, loss = raw_model(x, torch.ones_like(x), labels=torch.zeros_like(x))
                scaler.scale(loss).backward()
                raw_model.zero_grad(set_to_none=True)
                del x, logits, loss
                torch.cuda.empty_cache()
                break
            except RuntimeError as e:
                if not is_oom(e) or mb == 1:
                    raise
                raw_model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                mb //= 2
        mb = int(ctx.all_min(mb))
        torch.cuda.reset_peak_memory_stats(ctx.device)
    eff = int(cfg.get("effective_batch", 32))
    accum = max(1, math.ceil(eff / (mb * ctx.world)))
    per_step = mb * ctx.world * accum
    n = len(xs)
    steps_per_epoch = max(1, math.ceil(n / per_step))
    epochs = epochs or cfg["epochs"]
    total_steps = max_steps or steps_per_epoch * epochs
    sched = make_scheduler(opt, total_steps, cfg.get("warmup_frac", 0.06))
    model = wrap_ddp(raw_model, ctx)
    order = EpochOrder(n, cfg["seed"])

    # ---- resume -------------------------------------------------------------------------------
    st = {"epoch": 0, "consumed": 0, "global_step": 0, "best_f1": -1.0, "best_epoch": -1,
          "bad_epochs": 0, "history": [], "skipped": 0, "train_seconds": 0.0, "tokens": 0}
    last_pt = os.path.join(ckpt_dir, "last.pt")
    if mode == "train" and resume and os.path.exists(last_pt):
        ck = torch_load(last_pt)
        load_trainable_state(raw_model, ck["trainable"])
        opt.load_state_dict(ck["optimizer"])
        sched.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        st.update(ck["state"])
        if ck.get("total_steps") != total_steps:
            log(ctx, run_dir, f"   [resume] total steps changed {ck.get('total_steps')} -> "
                              f"{total_steps} (micro-batch/world change); schedule rescaled")
        log(ctx, run_dir, f"   [resume] epoch {st['epoch']} consumed {st['consumed']} "
                          f"step {st['global_step']} best F1 {st['best_f1']:.4f}")

    def save_last():
        if not ctx.main:
            return
        tmp = last_pt + ".tmp"
        torch.save({"trainable": trainable_state_dict(raw_model), "optimizer": opt.state_dict(),
                    "scheduler": sched.state_dict(), "scaler": scaler.state_dict(),
                    "state": st, "total_steps": total_steps,
                    "micro_batch": mb, "world": ctx.world}, tmp)
        os.replace(tmp, last_pt)

    def save_best():
        if not ctx.main:
            return
        p = os.path.join(ckpt_dir, "best.pt")
        torch.save(trainable_state_dict(raw_model, half=half_best), p + ".tmp")
        os.replace(p + ".tmp", p)

    deadline = float(cfg.get("deadline_unix", 0) or 0)
    margin = float(cfg.get("deadline_margin_s", 300))
    ckpt_every = float(cfg.get("ckpt_every_min", 30)) * 60
    last_ckpt_t = time.time()
    t_start = time.time()
    clip = float(cfg.get("grad_clip", 1.0))
    log_every = int(cfg.get("log_every", 50))
    losses = []
    first_loss = None
    stop_reason = None
    tok_count, tok_t0 = 0, time.time()
    min_steps_for_watchdog = min(100, max(1, total_steps // 3))
    watchdog_done = st["global_step"] >= min_steps_for_watchdog

    model.train()
    while st["epoch"] < epochs and stop_reason is None:
        idx = order.indices(st["epoch"], ctx.rank, ctx.world, st["consumed"])
        pos = 0
        while pos < len(idx):
            if max_steps and st["global_step"] >= max_steps:
                stop_reason = "max_steps"
                break
            # ---------------- one optimizer step (accum micro-batches) ----------------
            step_loss = 0.0
            micro = 0
            for a in range(accum):
                chunk = idx[pos:pos + mb]
                if not chunk:
                    break
                pos += len(chunk)
                micro += 1
                x, y, m = collate(xs, ys, chunk, meta["pad_id"])
                x, y, m = x.to(ctx.device), y.to(ctx.device), m.to(ctx.device)
                sync = (a == accum - 1) or pos >= len(idx)
                cm = model.no_sync() if (ctx.world > 1 and not sync) else _null()
                try:
                    with cm:
                        with amp_ctx():
                            logits, loss = model(x, m, labels=y, class_weights=cw)
                            if teacher is not None:
                                with torch.no_grad():
                                    t_logits, _ = teacher(x, m)
                                loss = kd_loss(logits, t_logits, y, cfg.get("kd_alpha", 0.5),
                                               cfg.get("kd_T", 2.0), cw)
                        scaler.scale(loss / accum).backward()
                except RuntimeError as e:
                    if not is_oom(e):
                        raise
                    if ctx.world > 1:
                        log(ctx, run_dir, "   [oom] out of memory under DDP -> exit for retry")
                        save_last()
                        raise SystemExit(EXIT_OOM)
                    opt.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    if mb == 1:
                        raise
                    mb //= 2
                    accum *= 2
                    log(ctx, run_dir, f"   [oom] micro-batch -> {mb}, accumulation -> {accum}")
                    pos -= len(chunk)
                    step_loss, micro = 0.0, 0
                    break
                lv = float(loss.detach())
                step_loss += lv / accum
                tok_count += int(m.sum())
            if micro == 0:
                continue
            # ---------------- step (NaN-safe; identical decision on all ranks) ----------------
            scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_([p for p in raw_model.parameters()
                                                    if p.requires_grad], clip)
            # gnorm is identical on every rank (DDP all-reduces the gradients), so the skip
            # decision is too. fp16: GradScaler itself skips overflow steps; the first ~30 steps
            # are its normal scale calibration and are not counted as failures.
            finite = bool(torch.isfinite(gnorm))
            if finite or amp_dtype == torch.float16:
                scaler.step(opt)
            if not finite and st["global_step"] >= 30:
                st["skipped"] += 1
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            st["global_step"] += 1
            st["consumed"] = min(n, st["consumed"] + per_step)
            if first_loss is None and math.isfinite(step_loss):
                first_loss = step_loss
            losses.append(step_loss)
            collapsed = amp_dtype == torch.float16 and scaler.get_scale() < 1e-4
            if collapsed or (st["skipped"] >= 5 and
                             st["skipped"] > 0.05 * max(1, st["global_step"] - 30)):
                save_last()
                log(ctx, run_dir, f"   [nan] {st['skipped']} steps with non-finite gradients "
                                  f"(loss scale {scaler.get_scale():.3g}) -> abort")
                raise SystemExit(EXIT_NAN)
            if st["global_step"] % log_every == 0 or st["global_step"] == 1:
                el = time.time() - tok_t0
                rec = {"step": st["global_step"], "epoch": st["epoch"],
                       "loss": float(np.mean(losses[-log_every:])),
                       "lr": opt.param_groups[0]["lr"], "grad_norm": float(gnorm),
                       "tokens_per_s": tok_count * ctx.world / max(el, 1e-9),
                       "peak_mem_gb": (torch.cuda.max_memory_allocated(ctx.device) / 1e9
                                       if ctx.device.type == "cuda" else 0.0),
                       "skipped": st["skipped"]}
                log(ctx, run_dir, f"   step {rec['step']:>5}/{total_steps} | ep {st['epoch']} | "
                                  f"loss {rec['loss']:.4f} | lr {rec['lr']:.2e} | gnorm "
                                  f"{rec['grad_norm']:.2f} | {rec['tokens_per_s']:,.0f} tok/s | "
                                  f"mem {rec['peak_mem_gb']:.1f} GB", rec if mode == "train" else None)
            # ---------------- time guard / periodic checkpoint ----------------
            if mode == "train":
                over = [1.0 if (deadline and time.time() > deadline - margin) else 0.0]
                over = ctx.bcast(over)[0]
                if over:
                    st["train_seconds"] += time.time() - t_start
                    save_last()
                    log(ctx, run_dir, "   [time] notebook time budget reached -> checkpoint saved")
                    raise SystemExit(EXIT_TIME)
                if time.time() - last_ckpt_t > ckpt_every:
                    save_last()
                    last_ckpt_t = time.time()
                    log(ctx, run_dir, f"   [ckpt] last.pt written at step {st['global_step']}")
        if stop_reason is not None:
            break
        # ---------------- end of epoch ----------------
        st["epoch"] += 1
        st["consumed"] = 0
        if not eval_each_epoch:
            continue
        f1 = -1.0
        if ctx.main:
            f1, _m = quick_dev_f1(raw_model, meta, docs, cfg, ctx.device, amp_ctx, snap)
            raw_model.train()
        f1 = ctx.bcast([f1])[0]
        improved = f1 > st["best_f1"]
        if improved:
            st["best_f1"], st["best_epoch"], st["bad_epochs"] = f1, st["epoch"], 0
            save_best()
        else:
            st["bad_epochs"] += 1
        st["history"].append({"epoch": st["epoch"], "dev_macro_f1": f1,
                              "train_loss": float(np.mean(losses[-steps_per_epoch:]))})
        log(ctx, run_dir, f"   == epoch {st['epoch']}/{epochs} | dev strict macro-F1 {f1:.4f} "
                          f"{'(best)' if improved else ''} | train loss "
                          f"{st['history'][-1]['train_loss']:.4f}",
            {"epoch_end": st["epoch"], "dev_macro_f1": f1} if mode == "train" else None)
        if mode == "train":
            if not watchdog_done and st["global_step"] >= min_steps_for_watchdog:
                watchdog_done = True
                if st["best_f1"] < float(cfg.get("watchdog_min_f1", 0.05)):
                    save_last()
                    log(ctx, run_dir, f"   [watchdog] dev F1 {st['best_f1']:.4f} < "
                                      f"{cfg.get('watchdog_min_f1', 0.05)} after "
                                      f"{st['global_step']} steps -> stop. Check data_report.json "
                                      f"(G1), the learning rate and logs/overfit.json.")
                    raise SystemExit(EXIT_EARLYFAIL)
            if st["bad_epochs"] >= int(cfg.get("patience", 3)):
                stop_reason = "early_stop"
            if time.time() - last_ckpt_t > 15 * 60:
                save_last()
                last_ckpt_t = time.time()
    st["train_seconds"] += time.time() - t_start
    el = time.time() - tok_t0
    summary = {"status": stop_reason or "done", "epochs_run": st["epoch"],
               "global_step": st["global_step"], "steps_per_epoch": steps_per_epoch,
               "total_steps": total_steps, "micro_batch": mb, "accum": accum, "world": ctx.world,
               "best_dev_macro_f1": st["best_f1"], "best_epoch": st["best_epoch"],
               "history": st["history"], "skipped_steps": st["skipped"],
               "first_loss": first_loss,
               "last_loss": float(np.mean(losses[-10:])) if losses else None,
               "tokens_per_s": tok_count * ctx.world / max(el, 1e-9),
               "seconds_per_step": el / max(1, len(losses)),
               "peak_mem_gb": (torch.cuda.max_memory_allocated(ctx.device) / 1e9
                               if ctx.device.type == "cuda" else 0.0),
               "trainable_params": count_params(raw_model, True),
               "total_params": count_params(raw_model), "train_seconds": st["train_seconds"],
               "lr_mult": lr_mult}
    return summary, raw_model


class _null:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


# =============================================================================================
# modes
# =============================================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", choices=["train", "smoke", "overfit", "probe"], default="train")
    ap.add_argument("--max-steps", type=int, default=None)
    ap.add_argument("--tag", default="")
    a = ap.parse_args()
    run_dir = a.run_dir
    cfg = read_json(os.path.join(run_dir, "logs", "run_config.json"))
    ctx = Ctx()
    code = 0
    try:
        meta, docs = load_cache(cfg["data_dir"])
        train_ids = meta["split"]["train"] + (meta["split"]["dev"] if cfg.get("refit") else [])
        tr_docs = [docs[d] for d in train_ids]
        xs, ys, cstats = build_train_chunks(tr_docs, cfg["chunk_len"], cfg["train_stride"],
                                            cfg.get("neg_keep", 0.3), cfg["seed"], meta["eos_id"])
        if ctx.main:
            print(f"   train chunks: {cstats}", flush=True)
        # first-batch assertions (section 9.4)
        x, y, m = collate(xs, ys, list(range(min(4, len(xs)))), meta["pad_id"])
        assert x.max().item() < meta["vocab_size"] and x.min().item() >= 0, "token id out of range"
        assert ((y == IGNORE) | ((y >= 0) & (y < 1 + 2 * len(meta["labels"])))).all(), "bad label"
        assert (y[m == 0] == IGNORE).all(), "labels on padding"
        assert (y[:, 0] == IGNORE).all() and (x[:, 0] == meta["eos_id"]).all(), "EOS slot wrong"
        assert x.shape[1] <= cfg["chunk_len"] <= 1024

        if a.mode == "smoke":
            steps = a.max_steps or 30
            s, _ = train_loop(cfg, ctx, run_dir, meta, docs, xs, ys, "smoke", max_steps=steps,
                              resume=False, eval_each_epoch=False)
            if ctx.main:
                epochs = cfg["epochs"]
                train_min = s["seconds_per_step"] * s["steps_per_epoch"] * epochs / 60
                n_dev_tok = sum(len(docs[d]["ids"]) for d in meta["split"]["dev"])
                eval_min = epochs * n_dev_tok / max(1.0, 3 * s["tokens_per_s"]) / 60
                s["projected_train_minutes"] = train_min + eval_min
                s["loss_decreased"] = (s["last_loss"] is not None and s["first_loss"] is not None
                                       and s["last_loss"] < s["first_loss"])
                write_json(os.path.join(run_dir, "logs", f"smoke{a.tag}.json"), s)
                print(f"   smoke ok: {s['tokens_per_s']:,.0f} tok/s | mem {s['peak_mem_gb']:.1f} GB"
                      f" | micro-batch {s['micro_batch']} x accum {s['accum']} x world "
                      f"{s['world']} | projected {s['projected_train_minutes']:.1f} min", flush=True)
        elif a.mode == "overfit":
            pos = [i for i, yy in enumerate(ys) if (yy > 0).any()][:64]
            c2 = dict(cfg)
            c2["dropout"] = 0.0
            c2["warmup_frac"] = 0.05
            # memorising 64 chunks is a pipeline check, not a tuning run: use a high LR
            mult = 10.0 if cfg["technique"] in FULL_PARAM else 3.0
            s, _ = train_loop(c2, ctx, run_dir, meta, docs, [xs[i] for i in pos],
                              [ys[i] for i in pos], "overfit", lr_mult=mult,
                              max_steps=a.max_steps or 150, epochs=10 ** 6, resume=False,
                              eval_each_epoch=False)
            if ctx.main:
                passed = s["last_loss"] < 0.05 or s["last_loss"] < 0.2 * s["first_loss"]
                s["passed"] = bool(passed)
                # skip the model only if it learned (almost) nothing -- a likely label bug
                s["fatal"] = bool(not passed and s["last_loss"] > 0.6 * s["first_loss"])
                write_json(os.path.join(run_dir, "logs", "overfit.json"), s)
                print(f"   overfit-64: loss {s['first_loss']:.3f} -> {s['last_loss']:.4f} "
                      f"({'PASS' if passed else 'FAIL'})", flush=True)
        elif a.mode == "probe":
            results = []
            spe = None
            for mult in cfg.get("probe_mults", [1 / 3, 1.0, 3.0]):
                budget = a.max_steps or cfg.get("probe_steps")
                if budget is None:
                    if spe is None:
                        eff = cfg.get("effective_batch", 32)
                        spe = max(1, math.ceil(len(xs) / eff))
                    budget = min(200, max(30, spe))
                s, model = train_loop(cfg, ctx, run_dir, meta, docs, xs, ys, "probe",
                                      lr_mult=mult, max_steps=budget, epochs=10 ** 6,
                                      resume=False, eval_each_epoch=False)
                f1 = -1.0
                if ctx.main:
                    _amp, amp_ctx = amp_setup(ctx.device)
                    f1, _ = quick_dev_f1(model, meta, docs, cfg, ctx.device, amp_ctx,
                                         SnapRules.from_json(meta.get("snap_rules") or {}))
                f1 = ctx.bcast([f1])[0]
                results.append({"lr_mult": mult, "lr": cfg["lr"] * mult, "steps": budget,
                                "dev_macro_f1": f1, "last_loss": s["last_loss"]})
                if ctx.main:
                    print(f"   probe lr x{mult:.3g} ({cfg['lr'] * mult:.2e}): dev F1 {f1:.4f} "
                          f"loss {s['last_loss']:.4f}", flush=True)
                del model
                if ctx.device.type == "cuda":
                    torch.cuda.empty_cache()
            best = max(results, key=lambda r: (r["dev_macro_f1"], -(r["last_loss"] or 1e9)))
            if ctx.main:
                write_json(os.path.join(run_dir, "logs", "lr_probe.json"),
                           {"best_lr_mult": best["lr_mult"], "best_lr": best["lr"],
                            "results": results})
        else:
            s, _ = train_loop(cfg, ctx, run_dir, meta, docs, xs, ys, "train")
            if ctx.main:
                s["chunk_stats"] = cstats
                write_json(os.path.join(run_dir, "logs", "train_summary.json"), s)
                if s["best_epoch"] < 0:
                    raise RuntimeError("training produced no best checkpoint")
                print(f"   training {s['status']}: best dev macro-F1 {s['best_dev_macro_f1']:.4f} "
                      f"at epoch {s['best_epoch']} ({s['train_seconds'] / 60:.1f} min)", flush=True)
    except SystemExit as e:
        code = int(e.code or 0)
    except Exception:
        traceback.print_exc()
        code = 1
    finally:
        try:
            ctx.close()
        except Exception:
            pass
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/validate_submission.py
"""
validate_submission.py -- block a bad Codabench upload before it costs a submission slot.

    python validate_submission.py PATH/submission.zip [--fold3 fold3-text.json] [--labels labels.txt]

Checks (doc sections 6.3, 6.4, 19):
  * the file is a zip whose ONLY member is `prediction.json`, at the root (no folder)
  * the keys are exactly the fold3-text.json ids (no missing, no extra), as strings
  * every span is [start, end, label] (an optional 4th string field is tolerated) with integer
    0 <= start < end <= len(text) and a label from labels.txt
  * spans in a document do not overlap
Without --fold3, the fold3 texts are taken from the default data cache.
"""
import argparse
import json
import sys
import zipfile


def validate(zip_path, texts, labels):
    errors, warnings = [], []
    n_spans = 0
    try:
        with zipfile.ZipFile(zip_path) as z:
            names = z.namelist()
            if names != ["prediction.json"]:
                errors.append(f"zip members must be exactly ['prediction.json'], got {names}")
            if "prediction.json" not in names:
                return {"ok": False, "errors": errors, "warnings": warnings, "n_spans": 0}
            pred = json.loads(z.read("prediction.json").decode("utf-8"))
    except zipfile.BadZipFile as e:
        return {"ok": False, "errors": [f"not a zip file: {e}"], "warnings": [], "n_spans": 0}
    if not isinstance(pred, dict):
        errors.append("prediction.json must be a JSON object {doc_id: [...]}")
        return {"ok": False, "errors": errors, "warnings": warnings, "n_spans": 0}
    want, got = set(map(str, texts)), set(pred)
    if want - got:
        errors.append(f"{len(want - got)} test ids missing, e.g. {sorted(want - got)[:5]}")
    if got - want:
        errors.append(f"{len(got - want)} unknown ids, e.g. {sorted(got - want)[:5]}")
    labset = set(labels)
    for did, spans in pred.items():
        if did not in texts:
            continue
        if not isinstance(spans, list):
            errors.append(f"{did}: value must be a list")
            continue
        n = len(texts[did])
        prev_end = -1
        for sp in sorted(spans, key=lambda x: (x[0], x[1]) if isinstance(x, list) and len(x) >= 2 else (0, 0)):
            n_spans += 1
            if not (isinstance(sp, list) and len(sp) in (3, 4)):
                errors.append(f"{did}: span must be [start, end, label]: {sp}")
                continue
            s, e, lab = sp[0], sp[1], sp[2]
            if not (isinstance(s, int) and isinstance(e, int)) or isinstance(s, bool):
                errors.append(f"{did}: offsets must be integers: {sp}")
                continue
            if not (0 <= s < e <= n):
                errors.append(f"{did}: bad offsets {sp} for a text of {n} chars")
            if lab not in labset:
                errors.append(f"{did}: unknown label {lab!r}")
            if len(sp) == 4 and not isinstance(sp[3], str):
                errors.append(f"{did}: the optional 4th field must be a string")
            if s < prev_end:
                errors.append(f"{did}: overlapping spans near {sp}")
            prev_end = max(prev_end, e)
    if n_spans == 0:
        warnings.append("no spans at all -- almost certainly a broken run")
    return {"ok": not errors, "errors": errors[:50], "warnings": warnings, "n_spans": n_spans,
            "n_docs": len(pred)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("zip")
    ap.add_argument("--fold3", default=None)
    ap.add_argument("--labels", default=None)
    ap.add_argument("--data-dir", default="/kaggle/working/phase2_data")
    a = ap.parse_args()
    if a.fold3:
        from ner_data import load_labels, load_texts
        texts = load_texts(a.fold3)
        labels = load_labels(a.labels) if a.labels else None
    else:
        from ner_data import load_cache
        meta, docs = load_cache(a.data_dir)
        texts = {docs[d]["orig_id"]: docs[d]["text"] for d in meta["split"]["test"]}
        labels = meta["labels"]
    if labels is None:
        from ner_data import EXPECTED_LABELS
        labels = EXPECTED_LABELS
    r = validate(a.zip, texts, labels)
    print(json.dumps(r, indent=2))
    sys.exit(0 if r["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/select_best.py
"""
select_best.py -- best-of-4 per experiment, and the model-coverage guard.

    python select_best.py --exp E1                 # (re)build OUT/E1_full_ft/BEST + summary
    python select_best.py --coverage               # which model won each notebook + extra uploads

Selection rule (section 6.1): highest dev strict macro-F1 (all 12 labels); ties -> fewer
trainable parameters. BEST/ gets a copy of the winner's submission.zip, best.pt, run_config.json
and DONE.json, plus SELECTION.json.

Coverage (section 10): the six submissions must involve different Phase-1 models. If the six
winners do not include all four models, list for each missing model its best submission across
techniques, to upload in addition.
"""
import argparse
import csv
import os
import shutil

from ner_common import (EXPERIMENTS, MODELS, REQUIRED_EXPS, default_input_roots,
                        iter_finished_runs, read_json, write_json)


def _model_rows(exp_dir):
    rows = []
    for mid, m in MODELS.items():
        d = os.path.join(exp_dir, m["folder"])
        done = read_json(os.path.join(d, "DONE.json"))
        if not done:
            continue
        tr = done.get("train") or {}
        rows.append({"model": mid, "arch": m["arch"], "dir": d,
                     "dev_macro_f1": done["dev_macro_f1"],
                     "dev_macro_precision": done.get("dev_macro_precision"),
                     "dev_macro_recall": done.get("dev_macro_recall"),
                     "dev_micro_f1": done.get("dev_micro_f1"),
                     "trainable_params": done.get("trainable_params"),
                     "total_params": done.get("total_params"),
                     "tokens_per_s": tr.get("tokens_per_s"),
                     "train_minutes": (tr.get("train_seconds") or 0) / 60,
                     "peak_mem_gb": tr.get("peak_mem_gb"),
                     "epochs_run": tr.get("epochs_run"), "best_epoch": tr.get("best_epoch")})
    return rows


def _key(r):
    """The selection rule: dev strict macro-F1, ties -> fewer trainable parameters."""
    return (r["dev_macro_f1"], -(r["trainable_params"] or 0))


def select(exp_dir, log=print):
    rows = _model_rows(exp_dir)
    if rows:
        cols = list(rows[0].keys())
        with open(os.path.join(exp_dir, "summary.csv"), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            w.writerows(rows)
        write_json(os.path.join(exp_dir, "summary.json"), rows)
    if not rows:
        log("   no finished models yet -- nothing to select")
        return None
    win = max(rows, key=_key)
    best = os.path.join(exp_dir, "BEST")
    os.makedirs(best, exist_ok=True)
    src = win["dir"]
    for rel, dst in (("submission/submission.zip", "submission.zip"),
                     ("checkpoints/best.pt", "best.pt"),
                     ("logs/run_config.json", "run_config.json"),
                     ("DONE.json", "DONE.json")):
        p = os.path.join(src, rel)
        if os.path.exists(p):
            shutil.copy2(p, os.path.join(best, dst))
    sel = {"winner": win["model"], "arch": win["arch"], "dev_macro_f1": win["dev_macro_f1"],
           "from": src, "n_models_finished": len(rows),
           "table": sorted(({k: r[k] for k in ("model", "dev_macro_f1", "trainable_params")}
                            for r in rows), key=lambda r: -r["dev_macro_f1"])}
    write_json(os.path.join(best, "SELECTION.json"), sel)
    log(f"   BEST of {len(rows)} -> {win['model']} ({win['arch']}), dev macro-F1 "
        f"{win['dev_macro_f1']:.4f}")
    return sel


def _exp_dirs(out_root, input_roots):
    """{exp_folder: {experiment dirs}} for every finished run found."""
    found = {}
    for exp_f, _mod_f, run_dir in iter_finished_runs(out_root, input_roots):
        found.setdefault(exp_f, set()).add(os.path.dirname(run_dir))
    return found


def coverage(out_root, input_roots, log=print):
    folders = {EXPERIMENTS[e]["folder"]: e for e in REQUIRED_EXPS}
    found = _exp_dirs(out_root, input_roots)
    winners, per_model_best = {}, {}
    for folder, exp in folders.items():
        best_row = None
        for d in sorted(found.get(folder, [])):
            for r in _model_rows(d):
                r["exp"] = exp
                r["zip"] = os.path.join(r["dir"], "submission", "submission.zip")
                if best_row is None or _key(r) > _key(best_row):
                    best_row = r
                pm = per_model_best.get(r["model"])
                if pm is None or _key(r) > _key(pm):
                    per_model_best[r["model"]] = r
        winners[exp] = best_row
    log("Winners (one upload per notebook):")
    for exp in REQUIRED_EXPS:
        w = winners[exp]
        log(f"   {exp}: " + (f"{w['model']} dev macro-F1 {w['dev_macro_f1']:.4f}  ->  {w['zip']}"
                             if w else "not finished yet"))
    used = {w["model"] for w in winners.values() if w}
    missing = [m for m in MODELS if m not in used]
    extra = []
    if missing:
        log(f"\nModels not represented among the winners: {missing}. Also upload:")
        for m in missing:
            r = per_model_best.get(m)
            if r:
                log(f"   {m}: its best run is {r['exp']} (dev macro-F1 {r['dev_macro_f1']:.4f}) "
                    f"->  {r['zip']}")
                extra.append({"model": m, "exp": r["exp"], "zip": r["zip"]})
            else:
                log(f"   {m}: no finished run anywhere yet")
    else:
        log("\nAll four Phase-1 models are represented among the winners.")
    n = sum(1 for w in winners.values() if w) + len(extra)
    log(f"\nPlanned uploads: {n} (max 10 per day, 40 per phase; each takes 15-30 min to score)")
    return {"winners": winners, "extra_uploads": extra}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", default=None)
    ap.add_argument("--out", default="/kaggle/working/phase2_out")
    ap.add_argument("--coverage", action="store_true")
    a = ap.parse_args()
    if a.coverage:
        res = coverage(a.out, default_input_roots())
        write_json(os.path.join(a.out, "coverage.json"), res)
        return
    folder = EXPERIMENTS.get(a.exp, {}).get("folder", a.exp)
    select(os.path.join(a.out, folder))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/aggregate.py
"""
aggregate.py -- the (technique x model) results table for the report.

    python aggregate.py [--out /kaggle/working/phase2_out] [--codabench scores.json]

Reads every phase2_out/*/*/DONE.json under --out and the attached input datasets, and writes
results.csv / results.md (+ per-label F1 table). `scores.json` is optional and filled in by
hand: {"E1/M4": 0.61, ...} with the Codabench strict macro-F1 of the uploads.
"""
import argparse
import csv
import os

from ner_common import EXPERIMENTS, MODELS, default_input_roots, iter_finished_runs, read_json

FOLDER_TO_EXP = {v["folder"]: k for k, v in EXPERIMENTS.items()}
FOLDER_TO_MODEL = {v["folder"]: k for k, v in MODELS.items()}


def collect(out_root, input_roots):
    rows = []
    for exp_folder, model_folder, run_dir in iter_finished_runs(out_root, input_roots):
        d = read_json(os.path.join(run_dir, "DONE.json"))
        rep = read_json(os.path.join(run_dir, "dev", "dev_report.json"), {})
        tr = d.get("train") or {}
        row = {"exp": FOLDER_TO_EXP.get(exp_folder, exp_folder),
               "technique": d.get("technique"), "model": FOLDER_TO_MODEL.get(model_folder, d.get("model")),
               "arch": d.get("arch"), "dev_macro_f1": d.get("dev_macro_f1"),
               "dev_macro_p": d.get("dev_macro_precision"), "dev_macro_r": d.get("dev_macro_recall"),
               "dev_micro_f1": d.get("dev_micro_f1"), "trainable_params": d.get("trainable_params"),
               "total_params": d.get("total_params"),
               "trainable_pct": (100 * d["trainable_params"] / d["total_params"]
                                 if d.get("trainable_params") and d.get("total_params") else None),
               "train_min": (tr.get("train_seconds") or 0) / 60, "epochs": tr.get("epochs_run"),
               "best_epoch": tr.get("best_epoch"), "train_tok_s": tr.get("tokens_per_s"),
               "infer_tok_s": d.get("inference_tokens_per_s"), "peak_mem_gb": tr.get("peak_mem_gb"),
               "decoding": f"{(d.get('decoding') or {}).get('mode')} "
                           f"{(d.get('decoding') or {}).get('o_shift')}"}
        per = (rep.get("metrics") or {}).get("per_label", {})
        for lab, v in per.items():
            row[f"f1_{lab}"] = v["f1"]
        rows.append(row)
    rows.sort(key=lambda r: (r["exp"], r["model"] or ""))
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="/kaggle/working/phase2_out")
    ap.add_argument("--codabench", default=None)
    a = ap.parse_args()
    rows = collect(a.out, default_input_roots())
    cb = read_json(a.codabench, {}) if a.codabench else {}
    for r in rows:
        r["codabench_macro_f1"] = cb.get(f"{r['exp']}/{r['model']}")
    if not rows:
        print("no finished runs found")
        return
    cols = list(dict.fromkeys(k for r in rows for k in r))
    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)
    main_cols = ["exp", "technique", "model", "dev_macro_f1", "dev_macro_p", "dev_macro_r",
                 "trainable_params", "trainable_pct", "train_min", "peak_mem_gb", "infer_tok_s",
                 "codabench_macro_f1"]

    def fmt(v):
        if isinstance(v, float):
            return f"{v:.4f}" if v < 10 else f"{v:,.1f}"
        if isinstance(v, int):
            return f"{v:,}"
        return "" if v is None else str(v)
    lines = ["| " + " | ".join(main_cols) + " |", "|" + "---|" * len(main_cols)]
    for r in rows:
        lines.append("| " + " | ".join(fmt(r.get(c)) for c in main_cols) + " |")
    labs = [c for c in cols if c.startswith("f1_")]
    if labs:
        lines += ["", "Per-label dev F1", "", "| exp | model | " + " | ".join(l[3:] for l in labs) + " |",
                  "|" + "---|" * (len(labs) + 2)]
        for r in rows:
            lines.append(f"| {r['exp']} | {r['model']} | " + " | ".join(fmt(r.get(l)) for l in labs) + " |")
    with open(os.path.join(a.out, "results.md"), "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines[:len(rows) + 2]))
    print(f"\nwritten: {os.path.join(a.out, 'results.csv')} and results.md")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase2/run_experiment.py
"""
run_experiment.py -- one notebook = one technique, trained on all four Phase-1 models.

    python run_experiment.py --exp E2                      # LoRMA on M4, M2, M3, M1
    python run_experiment.py --exp E1 --models M2,M3       # split a notebook across accounts
    python run_experiment.py --exp X3_crf --technique full_ft --head crf --models M4  # an extra

Per model: [restore earlier outputs] -> pre-flight (2-GPU smoke test with fallback to 1 GPU,
projected minutes) -> overfit-64 (first model) -> LR probe -> train -> dev report + fold3
prediction + validated submission.zip -> cleanup (last.pt deleted) -> DONE.json.
Then best-of-4 selection into BEST/ (select_best.py) and INDEX.md.

This process never touches the GPU itself: every GPU job is a subprocess, so a crash or a
CUDA error in one model cannot take down the whole notebook.
"""
import argparse
import os
import shutil
import signal
import subprocess
import sys
import threading
import time

from ner_common import (DEFAULT_EPOCHS, DEFAULT_LR, DEFAULT_MODEL_ORDER, EXPERIMENTS, MODELS,
                        STUDENT_LAYERS, DiskFull, default_input_roots, dir_size_gb, disk_guard,
                        find_files, find_phase1_models, free_gb, read_json, write_json)

HERE = os.path.dirname(os.path.abspath(__file__))
TRAIN = os.path.join(HERE, "ner_train.py")
PREDICT = os.path.join(HERE, "ner_predict.py")


# =============================================================================================
# subprocess helpers
# =============================================================================================
def run_cmd(cmd, log_path, env=None, timeout=None):
    """Stream a subprocess's output to the notebook and a log file, with a hard wall-clock
    timeout (a hung DDP job prints nothing, so the timeout must not depend on output).
    Returns (exit_code, timed_out); exit_code is -9 on timeout."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    full_env = dict(os.environ)
    full_env["PYTHONPATH"] = HERE + os.pathsep + full_env.get("PYTHONPATH", "")
    full_env.setdefault("PYTHONUNBUFFERED", "1")
    full_env.setdefault("NCCL_IB_DISABLE", "1")
    full_env.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
    full_env.setdefault("OMP_NUM_THREADS", "2")
    if env:
        full_env.update(env)
    lf = open(log_path, "a", encoding="utf-8")
    lf.write(f"\n$ {' '.join(cmd)}\n")
    lf.flush()
    p = subprocess.Popen(cmd, cwd=HERE, env=full_env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)

    def pump():
        for line in p.stdout:
            print(line, end="", flush=True)
            lf.write(line)
            lf.flush()

    th = threading.Thread(target=pump, daemon=True)
    th.start()
    timed_out = False
    try:
        code = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        try:
            os.killpg(p.pid, signal.SIGKILL)            # kills torchrun AND its workers
        except OSError:
            pass
        p.wait()
        code = -9
    th.join(timeout=10)
    lf.close()
    return code, timed_out


def gpu_count():
    try:
        out = subprocess.run([sys.executable, "-c", "import torch;print(torch.cuda.device_count())"],
                             capture_output=True, text=True, timeout=120)
        return int(out.stdout.strip().splitlines()[-1])
    except Exception:
        return 0


def train_cmd(run_dir, mode, nproc, extra=()):
    base = ["--run-dir", run_dir, "--mode", mode, *extra]
    if nproc > 1:
        return [sys.executable, "-m", "torch.distributed.run", "--standalone",
                "--nproc_per_node", str(nproc), TRAIN, *base]
    return [sys.executable, TRAIN, *base]


# =============================================================================================
# configs
# =============================================================================================
def student_layers(mid, phase1_dir):
    """Every second layer of the Phase-1 model (= STUDENT_LAYERS for the real 12/14-layer
    models; derived from config.json so tiny test models work too)."""
    cfg = read_json(os.path.join(phase1_dir, "config.json"))["config"]
    n = cfg.get("n_layer", cfg.get("num_hidden_layers"))
    ids = list(range(0, n, 2))
    if n in (12, 14):
        assert ids == STUDENT_LAYERS[mid], (ids, STUDENT_LAYERS[mid])
    return ids


def make_run_config(a, exp, folder, mid, technique, meta, data_dir, phase1_dir, deadline,
                    teacher_dir):
    lr = a.lr or DEFAULT_LR[technique]
    full_param = technique in ("full_ft", "kd", "student_hard")
    return {
        "exp": exp, "exp_folder": folder, "model": mid, "arch": MODELS[mid]["arch"],
        "kind": MODELS[mid]["kind"], "technique": technique, "phase1_dir": phase1_dir,
        "data_dir": data_dir, "labels": meta["labels"], "n_labels": len(meta["labels"]),
        "attn_mode": a.attn_mode, "head": a.head, "chunk_len": a.chunk_len,
        "train_stride": a.train_stride, "infer_stride": a.infer_stride, "neg_keep": a.neg_keep,
        "class_weight": a.class_weight, "lr": lr, "head_lr": 10 * lr if full_param else 1e-3,
        "weight_decay": 0.01, "warmup_frac": 0.06, "grad_clip": 1.0, "dropout": 0.1,
        "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "lorma_r": 16, "lorma_alpha": 32,
        "adapter_bottleneck": 64, "partial_top_k": 6, "layer_decay": 0.85,
        "kd_alpha": 0.5, "kd_T": 2.0, "teacher_dir": teacher_dir,
        "layer_ids": student_layers(mid, phase1_dir) if technique in ("kd", "student_hard") else None,
        "epochs": a.epochs or DEFAULT_EPOCHS[technique], "patience": a.patience,
        "effective_batch": a.effective_batch, "micro_batch": a.micro_batch, "eval_batch": 32,
        "seed": 42, "ckpt_every_min": a.ckpt_every_min, "deadline_unix": deadline,
        "deadline_margin_s": 300, "refit": a.refit,
        # PEFT methods can start slower; a false early stop would waste the whole run
        "watchdog_min_f1": (a.watchdog_min_f1 if technique in ("full_ft", "kd", "student_hard",
                                                               "partial_ft")
                            else min(a.watchdog_min_f1, 0.02)),
        "log_every": a.log_every, "probe_steps": a.probe_steps, "enforce_size": not a.cpu_test,
    }


# =============================================================================================
# restore outputs of earlier sessions (attached as input datasets)
# =============================================================================================
def restore_from_inputs(exp_folder, out_root, roots, log):
    """Copy finished (DONE.json) or resumable (last.pt) model folders of this experiment from
    attached datasets into the working output dir, unless already present."""
    for done in find_files("DONE.json", roots, max_hits=1000) + \
            find_files("last.pt", roots, max_hits=1000):
        parts = done.split(os.sep)
        if "phase2_out" not in parts or exp_folder not in parts:
            continue
        i = parts.index(exp_folder)
        if i + 1 >= len(parts) - 1:
            continue
        model_folder = parts[i + 1]
        src = os.sep.join(parts[:i + 2])
        dst = os.path.join(out_root, exp_folder, model_folder)
        if model_folder == "BEST" or os.path.exists(os.path.join(dst, "DONE.json")):
            continue
        if os.path.exists(dst) and not os.path.exists(os.path.join(src, "DONE.json")):
            continue                                    # never overwrite local progress
        log(f"   restoring {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)


def find_teacher(out_root, roots):
    cands = [os.path.join(out_root, "E1_full_ft", "BEST")]
    for s in find_files("SELECTION.json", roots, max_hits=1000):
        if os.sep + "E1_full_ft" + os.sep + "BEST" in s:
            cands.append(os.path.dirname(s))
    for c in cands:
        if all(os.path.exists(os.path.join(c, f)) for f in ("best.pt", "run_config.json")):
            return c
    return None


# =============================================================================================
# INDEX.md
# =============================================================================================
def write_index(out_root, exp_folder, records, next_cmd, a):
    st_path = os.path.join(out_root, "index_state.json")
    state = read_json(st_path, {})
    state[exp_folder] = {"records": records, "next": next_cmd,
                         "updated": time.strftime("%Y-%m-%d %H:%M:%S")}
    write_json(st_path, state)
    lines = ["# Phase-2 outputs (generated by run_experiment.py)", "",
             f"Output size: {dir_size_gb(out_root):.2f} GB | free disk: {free_gb(out_root):.1f} GB",
             ""]
    for ef in sorted(state):
        s = state[ef]
        lines += [f"## {ef}  (updated {s['updated']})", "",
                  "| model | status | dev macro-F1 | minutes | note |", "|---|---|---|---|---|"]
        for r in s["records"]:
            f1 = f"{r['dev_macro_f1']:.4f}" if isinstance(r.get("dev_macro_f1"), float) else ""
            mins = f"{r['minutes']:.1f}" if isinstance(r.get("minutes"), (int, float)) else ""
            lines.append(f"| {r['model']} | {r['status']} | {f1} | {mins} | {r.get('note', '')} |")
        sel = read_json(os.path.join(out_root, ef, "BEST", "SELECTION.json"))
        if sel:
            lines += ["", f"**Upload:** `{ef}/BEST/submission.zip` (winner {sel['winner']}, "
                          f"dev macro-F1 {sel['dev_macro_f1']:.4f})"]
        if s.get("next"):
            lines += ["", f"**Next:** `{s['next']}`"]
        lines.append("")
    lines += ["Save this session's output (Save Version, or Output -> New Dataset) before it "
              "ends; attach it to the next session so finished models are skipped and "
              "unfinished ones resume.", ""]
    with open(os.path.join(out_root, "INDEX.md"), "w") as f:
        f.write("\n".join(lines))


# =============================================================================================
# main loop
# =============================================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", required=True, help="E1..E6, X1, X2, or a custom name with --technique")
    ap.add_argument("--technique", default=None)
    ap.add_argument("--models", default=None, help="comma list, default M4,M2,M3,M1")
    ap.add_argument("--nproc", default="auto", choices=["auto", "1", "2"])
    ap.add_argument("--resume", default="auto", choices=["auto", "none"])
    ap.add_argument("--time-budget-h", type=float, default=11.0)
    ap.add_argument("--out", default="/kaggle/working/phase2_out")
    ap.add_argument("--data-dir", default="/kaggle/working/phase2_data")
    ap.add_argument("--attn-mode", default="bidir", choices=["bidir", "causal"])
    ap.add_argument("--head", default="linear", choices=["linear", "crf"])
    ap.add_argument("--chunk-len", type=int, default=512)
    ap.add_argument("--train-stride", type=int, default=384)
    ap.add_argument("--infer-stride", type=int, default=256)
    ap.add_argument("--neg-keep", type=float, default=0.3)
    ap.add_argument("--class-weight", default="none", choices=["none", "sqrt_inv"])
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--effective-batch", type=int, default=32)
    ap.add_argument("--micro-batch", type=int, default=16)
    ap.add_argument("--ckpt-every-min", type=float, default=30)
    ap.add_argument("--watchdog-min-f1", type=float, default=0.05)
    ap.add_argument("--probe-steps", type=int, default=None)
    ap.add_argument("--skip-probe", action="store_true")
    ap.add_argument("--skip-overfit", action="store_true")
    ap.add_argument("--refit", action="store_true", help="train on train+dev (extra X6)")
    ap.add_argument("--teacher-dir", default=None)
    ap.add_argument("--smoke-timeout-s", type=int, default=420)
    ap.add_argument("--min-free-gb", type=float, default=4.0)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--cpu-test", action="store_true", help="tiny-model CI mode")
    a = ap.parse_args()

    t_start = time.time()
    deadline = t_start + a.time_budget_h * 3600
    exp = a.exp
    spec = EXPERIMENTS.get(exp, {})
    technique = a.technique or spec.get("technique")
    if technique is None:
        sys.exit(f"unknown experiment {exp!r}; give --technique for a custom extra")
    folder = spec.get("folder") if not a.technique else exp
    folder = folder or exp
    models = [m.strip() for m in (a.models or ",".join(spec.get("models", DEFAULT_MODEL_ORDER))).split(",")]
    out_root = os.path.abspath(a.out)
    exp_dir = os.path.join(out_root, folder)
    os.makedirs(exp_dir, exist_ok=True)
    console = os.path.join(exp_dir, "console.log")
    roots = default_input_roots()

    def say(msg):
        print(msg, flush=True)
        with open(console, "a", encoding="utf-8") as f:
            f.write(time.strftime("%H:%M:%S ") + msg + "\n")

    say(f"== {folder}: technique {technique}, models {models}, budget {a.time_budget_h} h")

    # ---- data (CPU) ---------------------------------------------------------------------------
    from ner_data import GateFailure, find_or_build_cache, load_cache
    try:
        data_dir = find_or_build_cache(os.path.abspath(a.data_dir), roots, log=say)
    except GateFailure as e:
        say(f"STOP AND ASK (data gate): {e}")
        sys.exit(2)
    meta, _docs = load_cache(data_dir)
    del _docs

    p1 = find_phase1_models(roots)
    say(f"   Phase-1 weights found for: {sorted(p1)}")
    teacher_dir = None
    if technique == "kd":
        teacher_dir = a.teacher_dir or find_teacher(out_root, roots)
        if not teacher_dir:
            say("E6 needs E1_full_ft/BEST (best.pt + run_config.json). Run E1 first, or attach "
                "its output dataset to this notebook.")
            write_index(out_root, folder, [], f"run E1 first, then: python run_experiment.py --exp {exp}", a)
            sys.exit(2)
        say(f"   KD teacher: {teacher_dir}")
    if a.resume == "auto":
        restore_from_inputs(folder, out_root, roots, say)

    n_gpu = 0 if a.cpu_test else gpu_count()
    say(f"   GPUs visible: {n_gpu}")
    records, next_cmd = [], None
    first_model = True
    stop_all = False
    for mid in models:
        rec = {"model": mid, "status": "pending"}
        records.append(rec)
        run_dir = os.path.join(exp_dir, MODELS[mid]["folder"])
        logs = os.path.join(run_dir, "logs")
        os.makedirs(logs, exist_ok=True)
        cons = os.path.join(logs, "console.log")
        t_model = time.time()
        done = read_json(os.path.join(run_dir, "DONE.json"))
        if done:
            rec.update(status="done (skipped, already finished)", dev_macro_f1=done["dev_macro_f1"])
            say(f"-- {mid}: already finished (dev macro-F1 {done['dev_macro_f1']:.4f}), skipping")
            continue
        if stop_all:
            rec.update(status="not started")
            continue
        if mid not in p1:
            rec.update(status="FAILED", note="Phase-1 weights not attached")
            say(f"-- {mid}: Phase-1 weights for {MODELS[mid]['arch']} not found, skipping")
            continue
        try:
            disk_guard(out_root, a.min_free_gb, say, keep=os.path.join(run_dir, "checkpoints"))
        except DiskFull as e:
            rec.update(status="STOPPED", note=str(e))
            say(f"STOP: {e}")
            stop_all = True
            continue
        if deadline - time.time() < 20 * 60:
            rec.update(status="not started", note="session time budget exhausted")
            stop_all = True
            continue
        say(f"\n-- {folder} / {mid} ({MODELS[mid]['arch']}) --")

        cfg_path = os.path.join(logs, "run_config.json")
        old = read_json(cfg_path)
        resuming = os.path.exists(os.path.join(run_dir, "checkpoints", "last.pt")) and old
        cfg = make_run_config(a, exp, folder, mid, technique, meta, data_dir, p1[mid], deadline,
                              teacher_dir)
        if resuming:                                     # keep probed lr etc., refresh paths
            for k in ("lr", "head_lr", "lr_probe"):
                if k in old:
                    cfg[k] = old[k]
        write_json(cfg_path, cfg)

        # ---- pre-flight: nproc decision + projection ------------------------------------------
        dec = read_json(os.path.join(logs, "nproc_decision.json"))
        nproc, env = 1, {}
        if a.nproc != "auto":
            nproc = int(a.nproc)
        if dec and a.nproc == "auto":
            nproc, env = dec["nproc"], dec.get("env", {})
        smoke = None
        if dec is None or not os.path.exists(os.path.join(logs, "smoke.json")):
            tries = []
            if a.nproc == "auto" and n_gpu >= 2:
                tries = [(2, {}), (2, {"NCCL_P2P_DISABLE": "1"}), (1, {})]
            else:
                tries = [(nproc, {})]
            for npr, ev in tries:
                code, to = run_cmd(train_cmd(run_dir, "smoke", npr), cons, ev, a.smoke_timeout_s)
                if code == 0 and os.path.exists(os.path.join(logs, "smoke.json")):
                    nproc, env = npr, ev
                    break
                say(f"   smoke test with {npr} process(es) {ev or ''} failed "
                    f"({'timeout' if to else f'exit {code}'})")
            else:
                rec.update(status="FAILED", note="pre-flight smoke test failed (see console.log)")
                first_model = False
                continue
            write_json(os.path.join(logs, "nproc_decision.json"),
                       {"nproc": nproc, "env": env, "gpus": n_gpu,
                        "reason": "first configuration whose smoke test passed"})
        smoke = read_json(os.path.join(logs, "smoke.json"), {})
        proj = float(smoke.get("projected_train_minutes", 0) or 0)
        probe_min = 0 if (a.skip_probe or os.path.exists(os.path.join(logs, "lr_probe.json"))) \
            else proj * 0.5
        need = (proj + probe_min) * 1.25 + 5
        left = (deadline - time.time()) / 60
        say(f"   pre-flight: {nproc} process(es) {env or ''} | projected {proj:.1f} min train "
            f"+ {probe_min:.1f} min probe | {left:.0f} min left in budget")
        if need > left and not resuming:
            rec.update(status="not started", note=f"needs ~{need:.0f} min, {left:.0f} left")
            next_cmd = (f"python run_experiment.py --exp {exp} --models "
                        f"{','.join(models[models.index(mid):])}")
            stop_all = True
            continue

        # ---- overfit-64 (first model of the notebook) ----------------------------------------
        if first_model and not a.skip_overfit:
            ov = read_json(os.path.join(logs, "overfit.json"))
            if ov is None:
                run_cmd(train_cmd(run_dir, "overfit", 1), cons)
                ov = read_json(os.path.join(logs, "overfit.json"))
            if ov is None or not ov.get("passed"):
                fatal = ov is None or ov.get("fatal")
                say(f"   overfit-64 {'FAILED' if ov else 'crashed'}"
                    f"{' -> skipping this model' if fatal else ' (partial progress: warning only)'}")
                if fatal:
                    rec.update(status="FAILED", note="overfit-64 failed: check G1 / labels")
                    first_model = False
                    continue
        first_model = False

        # ---- LR probe -----------------------------------------------------------------------------
        if not a.skip_probe and not resuming:
            pr = read_json(os.path.join(logs, "lr_probe.json"))
            if pr is None:
                code, _ = run_cmd(train_cmd(run_dir, "probe", nproc), cons, env)
                pr = read_json(os.path.join(logs, "lr_probe.json"))
                if pr is None and nproc > 1:
                    code, _ = run_cmd(train_cmd(run_dir, "probe", 1), cons)
                    pr = read_json(os.path.join(logs, "lr_probe.json"))
            if pr:
                cfg["lr"] = pr["best_lr"]
                if technique in ("full_ft", "kd", "student_hard"):
                    cfg["head_lr"] = 10 * cfg["lr"]
                cfg["lr_probe"] = pr
                write_json(cfg_path, cfg)
                say(f"   LR probe -> lr {cfg['lr']:.2e}")
            else:
                say("   LR probe failed; keeping the default learning rate")

        # ---- train --------------------------------------------------------------------------------
        code, _ = run_cmd(train_cmd(run_dir, "train", nproc), cons, env)
        if code not in (0, 2, 4, 5) and nproc > 1:
            say(f"   training exited with {code} under DDP -> retrying on 1 GPU (resumes)")
            write_json(os.path.join(logs, "nproc_decision.json"),
                       {"nproc": 1, "env": {}, "gpus": n_gpu,
                        "reason": f"DDP training failed with exit {code}"})
            code, _ = run_cmd(train_cmd(run_dir, "train", 1), cons)
        if code == 2:
            rec.update(status="PAUSED", note="time budget; last.pt saved")
            next_cmd = (f"python run_experiment.py --exp {exp} --models "
                        f"{','.join(models[models.index(mid):])}   (attach this output first)")
            stop_all = True
            continue
        if code != 0:
            why = {4: "non-finite gradients (try a lower --lr)",
                   5: "early-fail watchdog (dev F1 ~ 0)",
                   3: "out of memory"}.get(code, f"exit {code}")
            rec.update(status="FAILED", note=f"training: {why}")
            continue

        # ---- predict + submission ----------------------------------------------------------------
        code, _ = run_cmd([sys.executable, PREDICT, "--run-dir", run_dir], cons, env)
        done = read_json(os.path.join(run_dir, "DONE.json"))
        if code != 0 or not done:
            rec.update(status="FAILED", note="prediction/submission step failed (best.pt kept)")
            continue
        for f in ("last.pt", "last.pt.tmp"):
            p = os.path.join(run_dir, "checkpoints", f)
            if os.path.exists(p):
                os.remove(p)
        mins = (time.time() - t_model) / 60
        rec.update(status="done", dev_macro_f1=done["dev_macro_f1"], minutes=mins)
        say(f"   {mid} finished in {mins:.1f} min | dev strict macro-F1 {done['dev_macro_f1']:.4f}"
            f" | output so far {dir_size_gb(out_root):.2f} GB")
        write_index(out_root, folder, records, next_cmd, a)

    # ---- best-of-4 ------------------------------------------------------------------------------
    from select_best import select
    sel = select(exp_dir, log=say)
    write_index(out_root, folder, records, next_cmd, a)
    say(f"\n== {folder} finished this session in {(time.time() - t_start) / 60:.1f} min")
    if sel:
        say(f"   UPLOAD: {os.path.join(exp_dir, 'BEST', 'submission.zip')} "
            f"(winner {sel['winner']}, dev macro-F1 {sel['dev_macro_f1']:.4f})")
    if next_cmd:
        say(f"   NEXT: {next_cmd}")
    say("   Remember to save this session's output (Save Version / Output -> New Dataset).")


if __name__ == "__main__":
    main()


In [ ]:
# every module must be byte-identical to the generated version (same code in all notebooks)
bad = []
for rel, h in EXPECTED_MD5.items():
    p = os.path.join("/kaggle/working/phase2", rel)
    if hashlib.md5(open(p, "rb").read()).hexdigest() != h:
        bad.append(rel)
assert not bad, f"modules differ from the generated version: {bad}"
print(f"all {len(EXPECTED_MD5)} modules verified")
sys.path.insert(0, "/kaggle/working/phase2")
from ner_common import find_phase1_models, find_files
print("Phase-1 weights:", json.dumps(find_phase1_models(), indent=1))
print("bpe_model.json:", find_files("bpe_model.json", ["/kaggle/input"])[:4])


In [ ]:
!cd /kaggle/working/phase2 && python run_experiment.py --exp E5 --nproc auto --resume auto

In [ ]:
print(open("/kaggle/working/phase2_out/INDEX.md").read()
      if os.path.exists("/kaggle/working/phase2_out/INDEX.md") else "no INDEX.md yet")
for dp, dn, fn in sorted(os.walk("/kaggle/working/phase2_out")):
    depth = dp.count(os.sep) - "/kaggle/working/phase2_out".count(os.sep)
    if depth <= 2:
        size = sum(os.path.getsize(os.path.join(dp, f)) for f in fn) / 1e6
        print("  " * depth + os.path.basename(dp) + f"/  ({size:.1f} MB in files here)")


**Run this notebook as a commit** (Save Version -> Save & Run All): it keeps running without the browser, and `/kaggle/working/phase2_out` becomes the notebook's output. Attach that output to the next session of this notebook to skip finished models / resume an unfinished one.

Upload `phase2_out/<experiment>/BEST/submission.zip` to Codabench, then run `python select_best.py --coverage` once all six are done.